# C++ 链表（Linked List）完全教程

> 从零基础到手写 std::list，为学习 STL list 容器打下坚实基础。

## 如何学习本教程

**前置知识：** 指针与引用、`struct`、构造函数、`new/delete`、函数返回指针、时间复杂度、`std::vector` 基础。

**学习路线：**

| 层级 | 建议章节 | 学习目标 |
| ---- | -------- | -------- |
| 必学 | 第 1-10 章 | 理解单链表节点、遍历、插入、删除、反转 |
| 必学 | 第 11-14 章 | 掌握连接、合并、环检测，并用类封装管理内存 |
| 选学 | 第 15-22 章 | 理解循环链表、双向链表和双向循环链表 |
| 进阶 | 第 23-26 章 | 过渡到模板、迭代器、`std::forward_list` 与 `std::list` |

**工程提醒：** 手写裸指针链表主要用于理解底层原理。实际项目优先使用标准库容器；若必须手写，应封装进类并遵循 RAII，避免让外部直接管理 owning raw pointer。

---

## 第一阶段：链表基础

### 1. 为什么需要动态数据结构？—— 链表的动机

#### 1.1 数组的局限性

数组是 C++ 中最基础的数据结构，但存在几个根本性的限制。

**问题一：固定数组的大小不可变**

In [ ]:
int arr[100];
// 必须在编译时确定大小

C 风格固定数组创建后容量不可变。如果实际只存了 5 个元素，浪费了 95 个位置；如果需要存 101 个元素，直接越界。

虽然 `std::vector` 能动态扩容，但扩容过程本身有代价：

In [ ]:
std::vector<int> v;
// 当 v.size() == v.capacity() 时，push_back 触发扩容：
// 1. 分配一块更大的新内存（通常是原来的 2 倍）
// 2. 把所有旧元素移动或复制到新内存
// 3. 释放旧内存

例如容量为 1000 时触发扩容，需要分配 2000 的新数组、移动或复制 1000 个元素、释放旧数组。单次 push_back 的时间复杂度从均摊 O(1) 瞬间跃升至 O(n)，在实时系统中可能无法接受。

**问题二：插入和删除需要移动大量元素**

假设数组 `[10, 20, 30, 40, 50]`，要在位置 1（值 20 之前）插入 15：

```text
初始状态:
  [10] [20] [30] [40] [50]
       插入位置

移动后:
  [10] [  ] [20] [30] [40] [50]

写入后:
  [10] [15] [20] [30] [40] [50]

含义:
  插入 15 前，20/30/40/50 都要后移。
```

移动了 4 个元素。如果数组有 n 个元素，在头部插入需要移动全部 n 个元素，时间复杂度 O(n)。

删除同理——删除位置 0 的元素后，剩余元素全部前移一位。

#### 1.2 链表如何解决这些问题

链表用截然不同的思路组织数据：

- **动态增长**：每次添加节点只分配一小块内存，无需预知总数，也不要求连续空间；但每个节点有指针开销，堆内存耗尽时分配会失败。
- **高效插入/删除**：已知头节点、前驱节点或有效迭代器时，通常只需修改少量指针，O(1) 完成，无需移动其他元素。若还需从头查找位置或前驱，整体仍为 O(n)。
- **内存独立分配**：节点可散布在内存任意位置，通过指针串联。

---

### 2. 链表概述

#### 2.1 什么是链表

链表是一种**线性数据结构**，由一系列节点组成，每个节点包含数据和指向下一个节点的指针。

**核心概念：逻辑相邻不等于物理相邻。**

这是理解链表的关键。数组中逻辑相邻的元素在内存中也相邻；链表中逻辑相邻的节点在内存中可能相距甚远。

以下通过示意图对比两者的内存布局：

**数组的内存布局：**

```text
逻辑顺序:
  [10] -> [20] -> [30] -> [40] -> [50]

物理内存:
  下标:   0      1      2      3      4
  内容: [10]   [20]   [30]   [40]   [50]
  地址: 1000   1004   1008   100C   1010

访问 arr[3]:
  0x1000 + 3 * 4 = 0x100C
  直接读到 [40]

含义:
  连续存放，所以能用下标算地址。
```

**链表的内存布局：**

```text
物理内存:
  0x0800: [40] next=0x2F00
  0x1500: [30] next=0x0800
  0x2000: [10] next=0x3A00
  0x2F00: [50] next=nullptr
  0x3A00: [20] next=0x1500

逻辑入口:
  head -> 0x2000[10] -> 0x3A00[20] -> 0x1500[30]
       -> 0x0800[40] -> 0x2F00[50] -> nullptr

含义:
  物理地址可乱序；逻辑顺序由 next 串起来。
```

链表节点散布在内存各处。`head` 记录首节点地址，每个节点的 `next` 记录后继地址，尾节点的 `next` 为 `nullptr`，标记链表终止。

#### 2.2 链表与数组的全方位对比表

| 维度 | 数组 / `std::vector` | 单链表 | 记忆点 |
| ---- | --------------------- | ------ | ------ |
| 内存布局 | 连续空间 | 节点分散 | 数组靠位置，链表靠指针 |
| 顺序依据 | 下标 | `head` + `next` | `head` 决定入口 |
| 随机访问 | O(1) | O(n) | 链表不能跳到第 i 个 |
| 头部插入 | O(n) | O(1) | 链表改 `head` 即可 |
| 中间插入 | O(n) | 已知前驱 O(1) | 找前驱仍要 O(n) |
| 尾部插入 | 均摊 O(1) | 有 `tail` O(1) | 无 `tail` 要遍历 |
| 删除节点 | O(n) | 已知前驱 O(1) | 单链表依赖前驱 |
| 查找值 | O(n) | O(n) | 都要逐个比 |
| 内存开销 | 较低 | 多 `next` | 小数据时指针很贵 |
| 缓存友好性 | 较好 | 较差 | 连续内存更快 |
| 容量变化 | 可能扩容复制 | 逐个分配 | 链表无固定容量 |
| 实现难度 | 下标和边界 | 指针和释放 | 链表更易出错 |

补充说明：`std::vector` 尾部插入均摊 O(1)，但扩容时单次为 O(n)。单链表若无尾指针，尾部插入需遍历全链，为 O(n)。有序数组可二分查找 O(log n)；有序链表因无法按下标跳转，仍需顺序查找。

#### 2.3 链表的优缺点分析

**优点：**

1. **动态大小**：长度随运行时增减，无需预先指定，不浪费空间。
2. **高效插入/删除**：已知头节点、前驱节点或有效迭代器时只需修改指针，O(1) 完成；数组则需移动大量元素。若需要先查找位置或前驱，整体仍是 O(n)。
3. **无需大块连续内存**：节点散布各处，降低了对连续空间的依赖；但仍可能因堆内存不足、碎片或分配器限制而分配失败。

**缺点：**

1. **不支持随机访问**：访问第 i 个元素需从 head 遍历 i 次，O(n)；数组用 `arr[i]` 即可，O(1)。
2. **额外内存开销**：每个节点需额外存储指针（64 位系统占 8 字节）。数据域较小时（如 int 仅 4 字节），指针开销占比高达 200%。
3. **缓存不友好**：CPU 缓存预取连续内存，数组天然受益；链表节点分散在堆中，缓存命中率低，遍历可能慢数倍。
4. **容易出错**：手动管理指针易产生内存泄漏、悬空指针等问题，调试困难。

#### 2.4 链表的应用场景

1. **频繁在头部或中间插入/删除的场景**：比如实现栈（push/pop 在头部）、任务队列。
2. **无法预估数据量的场景**：比如实时接收数据流，不知道会有多少条数据到来。
3. **实现其他数据结构**：栈、队列、哈希表的链地址法（解决哈希冲突）、图的邻接表表示。
4. **操作系统的内存管理**：操作系统用链表管理空闲内存块。
5. **浏览器的历史记录**：前进/后退功能可以用双向链表实现。
6. **音乐播放器的播放列表**：上一首/下一首，双向链表天然适配。

---

### 3. 节点（Node）的定义

#### 3.1 数据域与指针域

每个节点由两部分组成：

```text
一个 Node 节点:

  [ data | next ]
            |
            v
      另一个 Node 或 nullptr

含义:
  data 存值；
  next 存“下一个节点在哪里”。
```

- **数据域（data）**：存储实际数据，类型可以是 int、double、string 甚至自定义结构体。
- **指针域（next）**：存储下一个节点的内存地址。尾节点的 next 为 `nullptr`（空指针），表示链表结束。

若干节点串联即构成完整的链表：

```text
head
  |
  v
[data:10 | next] -> [data:20 | next] -> [data:30 | next] -> nullptr
```

#### 3.2 用 struct 定义节点

在 C++ 中，我们用 `struct` 来定义节点类型：

In [ ]:
struct Node {
    int data;       // 数据域，存储整数值
    Node* next;     // 指针域，指向下一个 Node
    Node(int val) : data(val), next(nullptr) {}
};

逐行解释：

- `int data;` —— 数据域，可替换为任何其他类型。
- `Node* next;` —— 指针域，类型为 `Node*`（指向 Node 的指针）。`Node` 在定义内部引用自身，称为**自引用结构（self-referential structure）**，在 C++ 中合法，因为编译器已知指针的大小。
- `Node(int val) : data(val), next(nullptr) {}` —— 构造函数，通过初始化列表将 `data` 设为 `val`、`next` 设为 `nullptr`，避免遗漏初始化。

使用示例：

In [ ]:
Node* n1 = new Node(10);
// 创建一个节点，data = 10, next = nullptr
Node* n2 = new Node(20);
// 创建一个节点，data = 20, next = nullptr
n1->next = n2;
// 把 n1 的 next 指向 n2

此时内存中的状态：

```text
变量保存的地址:
  n1 = 0x1000
  n2 = 0x2000

指针关系:
  n1
   |
   v
  0x1000 [data:10 | next] -> 0x2000 [data:20 | next] -> nullptr

含义:
  n1->next 保存的是 n2 指向的那个节点地址。
```

#### 3.3 节点的内存分配（new / delete）

**创建节点用 new：**

In [ ]:
Node* newNode = new Node(42);

`new Node(42)` 做了两件事：
1. 在堆（heap）上分配一块能容纳 `Node` 结构体的内存
2. 调用构造函数，将 data 设为 42，next 设为 nullptr

返回的是这块内存的地址，赋给指针 `newNode`。

**销毁节点用 delete：**

In [ ]:
delete newNode;
// 释放这块内存
newNode = nullptr;
// 好习惯：delete 后置空指针，防止误用

`delete` 做了两件事：
1. 调用对象的析构函数（当前 `Node` 只有 `int` 和指针成员，析构本身没有额外工作；若数据域是 `std::string` 等对象，成员析构仍会发生）
2. 释放堆上的内存

**内存泄漏警告：**

In [ ]:
Node* p = new Node(10);
p = new Node(20);
// 危险！第一个节点的地址丢失了，内存泄漏！
// 第一个节点占的内存再也无法释放，直到程序结束

正确做法：

In [ ]:
Node* p = new Node(10);
delete p;
// 先释放旧节点
p = new Node(20);
// 再分配新节点

**基本纪律**：每 `new` 一个节点，最终都要对应 `delete`。

---

## 第二阶段：单链表基本操作

以下操作均基于此 Node 定义，`head` 指向首节点，空链表时 `head == nullptr`：

In [ ]:
struct Node {
    int data;
    Node* next;
    Node(int val) : data(val), next(nullptr) {}
};

---

### 4. 链表的显示与遍历

遍历是链表最基本的操作——沿 next 指针从头走到尾，访问每个节点。

#### 4.1 迭代显示链表

**概念解释：**

遍历需要一个临时"游标"指针 `curr`，从 head 出发逐步后移。不要移动真正负责保存入口的 `head` 指针——一旦入口丢失，前面的节点就再也找不回来了。

**思路分析：**

1. 创建临时指针 `curr`，指向 `head`
2. 当 `curr` 不为 `nullptr` 时，循环：
   - 打印 `curr->data`
   - 将 `curr` 移动到 `curr->next`
3. `curr` 为 `nullptr` 时，到达链表末尾，结束

**代码实现：**

In [ ]:
void display(Node* head) {
    Node* curr = head;          // 临时指针，从 head 开始
    while (curr != nullptr) {   // 还有节点没访问
        std::cout << curr->data << " -> ";
        curr = curr->next;      // 移动到下一个节点
    }
    std::cout << "NULL" << std::endl;
}

**逐步图解：**

假设链表如下：

```text
head
  |
  v
[10] -> [20] -> [30] -> nullptr
```

执行 `display(head)` 的过程：

```text
初始: p -> [10], head 不动

第 1 步: 输出 10, p -> [20]
第 2 步: 输出 20, p -> [30]
第 3 步: 输出 30, p -> nullptr, 循环结束

最终输出: 10 -> 20 -> 30 -> NULL
head 始终未移动，链表结构完好。
```

**完整可运行示例：**

In [ ]:
#include <iostream>

In [ ]:
struct Node {
    int data;
    Node* next;
    Node(int val) : data(val), next(nullptr) {}
};

In [ ]:
void display(Node* head) {
    Node* curr = head;
    while (curr != nullptr) {
        std::cout << curr->data << " -> ";
        curr = curr->next;
    }
    std::cout << "NULL" << std::endl;
}

In [ ]:
int main() {
    // 手动构建链表: 10 -> 20 -> 30 -> NULL
    Node* head = new Node(10);
    head->next = new Node(20);
    head->next->next = new Node(30);

    display(head);  // 输出: 10 -> 20 -> 30 -> NULL

    // 释放内存
    Node* curr = head;
    while (curr != nullptr) {
        Node* temp = curr;
        curr = curr->next;
        delete temp;
    }

    return 0;
}

In [ ]:
main();

**复杂度分析：**
- 时间复杂度：O(n)，需要遍历所有 n 个节点
- 空间复杂度：O(1)，只用了一个额外的指针 `curr`

#### 4.1.1 用 for 循环创建链表（推荐方法）

**概念解释：**

上面的示例是手动逐个创建节点、逐个连接，当数据量大时非常繁琐。实际开发中通常用 **for 循环从数组或其他数据源批量创建链表**。这样做不仅代码简洁，还避免了手写时的错误。

**思路分析：**

给定一个数组 `arr[]`，用 for 循环创建链表的基本步骤：

1. 初始化：创建第一个节点 `head`，用指针 `tail` 记录链表的尾部（方便逐个追加）
2. 循环：从数组的第 2 个元素（下标 1）开始，逐个创建新节点，链接到 `tail->next`，然后 `tail` 后移
3. 结束：最后一个节点的 `next` 自动为 `nullptr`

**代码实现：**

In [ ]:
// 从数组创建链表

In [ ]:
Node* createFromArray(const int arr[], int size) {
    if (size <= 0) return nullptr;  // 空数组或非法长度，返回空链表
    
    Node* head = new Node(arr[0]);  // 创建首节点
    Node* tail = head;              // tail 跟踪尾部
    
    for (int i = 1; i < size; i++) {
        Node* newNode = new Node(arr[i]);
        tail->next = newNode;        // 尾部指向新节点
        tail = newNode;              // tail 后移
    }
    
    return head;
}

**逐步图解：**

假设数组 `arr[] = {10, 20, 30}`，逐步创建链表的过程：

```text
初始化:
  arr[0] = 10
  创建 head
  head -> [10 | next] -> nullptr
  tail 指向 [10]

第一次迭代 (i=1):
  创建新节点 newNode = [20]
  tail->next = newNode
  tail = newNode (后移)
  
  head -> [10 | next] -> [20 | next] -> nullptr
                          ^
                         tail

第二次迭代 (i=2):
  创建新节点 newNode = [30]
  tail->next = newNode
  tail = newNode (后移)
  
  head -> [10 | next] -> [20 | next] -> [30 | next] -> nullptr
                                         ^
                                        tail

返回 head，链表构建完成。
```

**完整可运行示例：**

In [ ]:
#include <iostream>

In [ ]:
struct Node {
    int data;
    Node* next;
    Node(int val) : data(val), next(nullptr) {}
};

In [ ]:
void display(Node* head) {
    Node* curr = head;
    while (curr != nullptr) {
        std::cout << curr->data << " -> ";
        curr = curr->next;
    }
    std::cout << "NULL" << std::endl;
}

In [ ]:
Node* createFromArray(const int arr[], int size) {
    if (size <= 0) return nullptr;
    
    Node* head = new Node(arr[0]);
    Node* tail = head;
    
    for (int i = 1; i < size; i++) {
        Node* newNode = new Node(arr[i]);
        tail->next = newNode;
        tail = newNode;
    }
    
    return head;
}

// 释放链表内存

In [ ]:
void deleteList(Node* head) {
    Node* curr = head;
    while (curr != nullptr) {
        Node* temp = curr;
        curr = curr->next;
        delete temp;
    }
}

In [ ]:
int main() {
    // 用数组创建链表
    int arr[] = {10, 20, 30, 40, 50};
    int size = 5;
    
    Node* head = createFromArray(arr, size);
    
    std::cout << "链表内容: ";
    display(head);  // 输出: 10 -> 20 -> 30 -> 40 -> 50 -> NULL
    
    deleteList(head);
    
    return 0;
}

In [ ]:
main();

**输出结果：**

```text
链表内容: 10 -> 20 -> 30 -> 40 -> 50 -> NULL
```

**复杂度分析：**
- 时间复杂度：O(n)，创建 n 个节点，每个节点一次操作
- 额外空间复杂度：O(1)，只使用了少量辅助指针（`head`、`tail`、`newNode`）
  （若把“新建出的链表本身”也计入空间占用，则总占用为 O(n)）

**常见变种：用标准库创建链表**

也可以用 `std::vector` 作为临时存储，再逐个节点创建：

In [ ]:
#include <vector>

In [ ]:
Node* createFromVector(const std::vector<int>& vec) {
    if (vec.empty()) return nullptr;
    
    Node* head = new Node(vec[0]);
    Node* tail = head;
    
    for (size_t i = 1; i < vec.size(); i++) {
        Node* newNode = new Node(vec[i]);
        tail->next = newNode;
        tail = newNode;
    }
    
    return head;
}

#### 4.2 链表的递归显示

**概念解释：**

用递归遍历链表的思路：先处理当前节点，再对剩余链表递归调用。

**思路分析：**

递归的两个要素：
1. **基准情况（base case）**：`curr == nullptr`，什么都不做，返回
2. **递归情况**：打印当前节点数据，然后对 `curr->next` 递归调用

**代码实现：**

In [ ]:
void displayRecursive(Node* curr) {
    if (curr == nullptr) {  // 基准情况：空链表，结束
        std::cout << "NULL" << std::endl;
        return;
    }
    std::cout << curr->data << " -> ";   // 处理当前节点
    displayRecursive(curr->next);         // 递归处理剩余链表
}

**逐步图解（调用栈展开）：**

假设链表 `10 -> 20 -> 30 -> NULL`：

```text
压栈阶段（每层先打印，再递归下一层）:

  displayRecursive([10])      打印 10 ->
    displayRecursive([20])    打印 20 ->
      displayRecursive([30])  打印 30 ->
        displayRecursive(nullptr) 打印 NULL
```

用调用栈的视角来看（栈顶在上）：

```text
压栈完成:
  栈顶 nullptr
       [30]
       [20]
  栈底 [10]

返回阶段:
  nullptr -> [30] -> [20] -> [10]

最终输出:
  10 -> 20 -> 30 -> NULL
```

**复杂度分析：**
- 时间复杂度：O(n)
- 空间复杂度：O(n)，因为递归调用栈的深度等于链表长度。如果链表很长（比如上万个节点），可能导致栈溢出（stack overflow）。这是递归版本的缺点。

#### 4.3 链表的反向递归显示

**概念解释：**

正序递归"先处理当前，再递归后面"；反向递归则反过来——先递归到末尾，在回溯时打印，即**后序遍历**思想。

**思路分析：**

1. 先递归调用 `displayReverse(curr->next)`，一路深入到链表末尾
2. 到达 `nullptr` 后开始回溯
3. 回溯时再打印当前节点的数据

**代码实现：**

In [ ]:
void displayReverse(Node* curr) {
    if (curr == nullptr) {  // 基准情况
        return;
    }
    displayReverse(curr->next);          // 先递归到末尾
    std::cout << curr->data << " -> ";   // 回溯时再打印
}

**逐步图解（调用栈展开）：**

```text
初始链表:
  head -> [10] -> [20] -> [30] -> nullptr
```

用栈帧图更清晰地展示回溯过程：

```text
压栈阶段:
  只调用下一层，不打印当前节点。
  displayReverse([10])
    displayReverse([20])
      displayReverse([30])
        displayReverse(nullptr)
回溯阶段:
  [30] 恢复执行，打印 30 ->
  [20] 恢复执行，打印 20 ->
  [10] 恢复执行，打印 10 ->
打印时机:
  递归调用返回之后。
```

链表可视为特殊的树（每个节点只有一个子节点），因此树的遍历思想完全适用。

**复杂度分析：**
- 时间复杂度：O(n)
- 空间复杂度：O(n)，递归栈深度等于链表长度

---

### 5. 链表的统计操作

#### 5.1 计算链表节点数（迭代法）

**概念解释：**

从头到尾遍历，每经过一个节点计数器加一。

**思路分析：**

1. 创建计数器 `count = 0`
2. 创建临时指针 `curr = head`
3. 遍历链表，每经过一个节点 `count++`
4. 返回 `count`

**代码实现：**

In [ ]:
int countNodes(Node* head) {
    int count = 0;
    Node* curr = head;
    while (curr != nullptr) {
        count++;
        curr = curr->next;
    }
    return count;
}

**逐步图解：**

| 步骤 | 当前 `p` | `count` |
| ---- | -------- | ------- |
| 初始 | `[10]` | 0 |
| 1 | `[10]` → 后移 | 1 |
| 2 | `[20]` → 后移 | 2 |
| 3 | `[30]` → 后移 | 3 |
| 结束 | `nullptr` | 3 |

**复杂度分析：**
- 时间复杂度：O(n)
- 空间复杂度：O(1)

#### 5.2 计算链表节点数（递归法）

**概念解释：**

递归思路：链表的节点数 = 1 + 剩余链表的节点数。基准情况：空链表节点数为 0。

**代码实现：**

In [ ]:
int countNodesRecursive(Node* curr) {
    if (curr == nullptr) {  // 基准情况：空链表
        return 0;
    }
    return 1 + countNodesRecursive(curr->next);  // 1 + 剩余节点数
}

**逐步图解（递归展开）：**

```text
压栈阶段:
  count([10])
    count([20])
      count([30])
        count(nullptr)

```

```text
返回阶段:
  count(nullptr) = 0
  count([30]) = 1 + 0 = 1
  count([20]) = 1 + 1 = 2
  count([10]) = 1 + 2 = 3

最终返回:
  3
```

**复杂度分析：**
- 时间复杂度：O(n)
- 空间复杂度：O(n)，递归栈深度

#### 5.3 链表所有元素求和（迭代 + 递归）

**概念解释：**

与计数类似，遍历时累加每个节点的 data。

**迭代版本：**

In [ ]:
int sumIterative(Node* head) {
    int total = 0;
    Node* curr = head;
    while (curr != nullptr) {
        total += curr->data;    // 把当前节点的值累加
        curr = curr->next;
    }
    return total;
}

**递归版本：**

In [ ]:
int sumRecursive(Node* curr) {
    if (curr == nullptr) {
        return 0;
    }
    return curr->data + sumRecursive(curr->next);
}

**逐步图解（递归版本）：**

```text
压栈阶段:
  sum([10])
    sum([20])
      sum([30])
        sum(nullptr)

返回阶段:
  sum(nullptr) = 0
  sum([30]) = 30 + 0 = 30
  sum([20]) = 20 + 30 = 50
  sum([10]) = 10 + 50 = 60

最终返回：60
```

**复杂度分析：**
- 时间复杂度：O(n)
- 空间复杂度：迭代版 O(1)，递归版 O(n)

#### 5.4 链表中的最大元素（迭代 + 递归）

**概念解释：**

遍历链表，维护当前最大值。需特别注意**空链表**没有最大值，必须特殊处理。

**迭代版本：**

In [ ]:
int findMax(Node* head) {
    if (head == nullptr) {
        // 空链表，没有最大值。这里用 INT_MIN 作为哨兵值。
        // 实际项目中，抛异常或返回 optional<int> 更安全。
        return INT_MIN;
    }
    int maxVal = head->data;    // 先假设第一个节点的值最大
    Node* curr = head->next;    // 从第二个节点开始比较
    while (curr != nullptr) {
        if (curr->data > maxVal) {
            maxVal = curr->data;  // 发现更大的值，更新
        }
        curr = curr->next;
    }
    return maxVal;
}

注意：需要 `#include <climits>` 才能使用 `INT_MIN`。
这里用 `INT_MIN` 是教学中的哨兵写法；如果链表真实最大值也可能是 `INT_MIN`，调用者无法区分“空链表”和“最大值为 `INT_MIN`”，工程代码更推荐 `std::optional<int>` 或抛异常。

**递归版本：**

In [ ]:
int findMaxRecursive(Node* curr) {
    if (curr == nullptr) {
        return INT_MIN;  // 基准情况：空节点返回最小整数
    }
    int restMax = findMaxRecursive(curr->next);  // 后面节点的最大值
    // 返回当前节点的值 和 后续最大值 中更大的那个
    return (curr->data > restMax) ? curr->data : restMax;
}

**逐步图解（迭代版本）：**

```text
初始状态:
  head -> [5] -> [20] -> [15] -> [30] -> [10] -> nullptr
  maxVal = 5

遍历指针:
  p 从第二个节点 [20] 开始比较。
```

| 步骤 | 当前 `p` | 比较结果 | `maxVal` |
| ---- | -------- | -------- | -------- |
| 初始 | `[5]` | 用头节点初始化 | 5 |
| 1 | `[20]` | 20 更大，更新 | 20 |
| 2 | `[15]` | 15 不更大 | 20 |
| 3 | `[30]` | 30 更大，更新 | 30 |
| 4 | `[10]` | 10 不更大 | 30 |
| 结束 | `NULL` | 停止循环 | 30 |

**逐步图解（递归版本，回溯求最大值的过程）：**

```text
压栈阶段:
  max([5]) -> max([20]) -> max([15])
          -> max([30]) -> max([10]) -> max(nullptr)

返回阶段:
  max(nullptr) = INT_MIN
  max([10]) = max(10, INT_MIN) = 10
  max([30]) = max(30, 10) = 30
  max([15]) = max(15, 30) = 30
  max([20]) = max(20, 30) = 30
  max([5])  = max(5, 30) = 30
```

**复杂度分析：**
- 时间复杂度：O(n)
- 空间复杂度：迭代版 O(1)，递归版 O(n)

---

### 6. 链表中的搜索

#### 6.1 线性搜索（迭代法）

**概念解释：**

链表不支持随机访问，搜索只能从头逐个比较，即**线性搜索（Linear Search）**。

**思路分析：**

1. 从 head 开始遍历
2. 每访问一个节点，比较其 data 是否等于目标值
3. 找到了返回该节点的指针（或位置），没找到返回 nullptr

**代码实现：**

In [ ]:
Node* search(Node* head, int key) {
    Node* curr = head;
    while (curr != nullptr) {
        if (curr->data == key) {
            return curr;       // 找到了，返回节点指针
        }
        curr = curr->next;
    }
    return nullptr;            // 遍历完都没找到
}

**逐步图解：**

```text
搜索 key = 30:
  head -> [10] -> [20] -> [30] -> [40] -> nullptr
  p    -> [10]

移动过程:
  p=[10]，不相等，后移
  p=[20]，不相等，后移
  p=[30]，命中，返回 p
```

| 步骤 | 当前 `p` | 与 `key=30` 比较 | 下一步 |
| ---- | -------- | ---------------- | ------ |
| 1 | `[10]` | 不相等 | 后移 |
| 2 | `[20]` | 不相等 | 后移 |
| 3 | `[30]` | 相等 | 返回 `p` |

命中时：

```text
命中时:
  head -> [10] -> [20] -> [30] -> [40] -> nullptr
  p    -> [30]

返回值:
  返回 [30] 这个节点的地址。
```

**复杂度分析：**
- 时间复杂度：O(n)，最坏情况（目标在末尾或不存在）需要遍历全部节点
- 空间复杂度：O(1)

#### 6.2 线性搜索（递归法）

**代码实现：**

In [ ]:
Node* searchRecursive(Node* curr, int key) {
    if (curr == nullptr) {          // 基准情况：没找到
        return nullptr;
    }
    if (curr->data == key) {        // 找到了
        return curr;
    }
    return searchRecursive(curr->next, key);  // 继续在剩余链表中搜索
}

**逐步图解：**

```text
搜索 key = 30:
  search([10])
    search([20])
      search([30]) 命中，返回 node_30

调用栈返回过程：
  search([30]) 返回 node_30
  search([20]) 返回 node_30
  search([10]) 返回 node_30

命中节点的地址不会被复制成新节点，只是同一个指针逐层向上传回。
```

**复杂度分析：**
- 时间复杂度：O(n)
- 空间复杂度：O(n)，递归栈深度

#### 6.3 改进搜索——移至头部法（Move to Head）

**概念解释：**

如果某元素被反复搜索，每次都从头遍历效率很低。移至头部法的核心思想：**每次找到节点后，将其移到链表头部**，下次搜索同一元素时第一次就能找到。

这利用了**时间局部性**原理——刚被访问的数据很可能很快又被访问。

**思路分析：**

1. 搜索目标 key
2. 如果找到了（且不是头节点），将其从原位置摘除
3. 将其插入到链表头部
4. 更新 head 指针

**代码实现：**

In [ ]:
Node* moveToHead(Node* head, int key) {
    if (head == nullptr || head->data == key) {
        return head;  // 空链表或已在头部，无需操作
    }

    Node* prev = nullptr;   // 记录 curr 的前一个节点
    Node* curr = head;

    // 查找 key
    while (curr != nullptr && curr->data != key) {
        prev = curr;
        curr = curr->next;
    }

    if (curr == nullptr) {
        return head;  // 没找到，返回原 head
    }

    // 把 curr 从原位置摘除
    prev->next = curr->next;

    // 把 curr 插入到头部
    curr->next = head;
    head = curr;

    return head;  // 返回新的 head
}

**逐步图解：**

假设链表 `10 -> 20 -> 30 -> 40 -> NULL`，搜索 key = 30：

```text
搜索命中后:
  head -> [10] -> [20] -> [30] -> [40] -> nullptr

关键指针:
  head -> [10]
  prev -> [20]
  curr -> [30]

此时：
- head 指向原第一个节点 node_10
- prev 指向 curr 的前驱 node_20
- curr 指向命中的节点 node_30
```

完整过程用简化箭头表示：

```text
步骤 1: prev->next = curr->next
  head -> [10] -> [20] -> [40] -> nullptr
  curr -> [30] -> [40]

步骤 2: curr->next = head
  curr -> [30] -> [10] -> [20] -> [40] -> nullptr

步骤 3: head = curr
  head -> [30] -> [10] -> [20] -> [40] -> nullptr
```

**复杂度分析：**
- 时间复杂度：O(n)，搜索仍然需要遍历
- 空间复杂度：O(1)
- **实际效果**：如果同一个元素被搜索 m 次，第一次 O(n)，后续 m-1 次都是 O(1)，总时间从 O(mn) 降为 O(n + m)。在元素访问有局部性的场景下效果显著。

调用时必须接住返回的新头指针：

In [ ]:
head = moveToHead(head, key);

#### 6.4 改进搜索——转置法（Transpose）

**概念解释：**

移至头部法的缺点：若搜索一个仅出现一次的元素，它会把原来头部的高频元素挤到后面，可能反而降低整体效率。

转置法更温和：**每次找到节点，只和前一个节点交换位置**。频繁搜索的元素会"缓慢地"向头部移动，不会一次性跳太远。

**思路分析：**

1. 搜索目标 key
2. 如果找到了（且不是头节点），将其与前驱节点交换位置
3. 交换方式：修改指针指向，而非交换数据

**代码实现：**

In [ ]:
Node* transpose(Node* head, int key) {
    if (head == nullptr || head->data == key) {
        return head;  // 空链表或已在头部
    }

    Node* prevPrev = nullptr;   // prev 的前一个节点
    Node* prev = nullptr;       // curr 的前一个节点
    Node* curr = head;

    // 查找 key，同时维护 curr 前面的两层节点
    while (curr != nullptr && curr->data != key) {
        prevPrev = prev;
        prev = curr;
        curr = curr->next;
    }

    if (curr == nullptr || prev == nullptr) {
        return head;  // 没找到，或 key 就是 head（已在最前）
    }

    // 把 curr 和 prev 交换位置（改指针，不交换数据）：
    // 之前: ... -> prevPrev -> prev -> curr -> curr->next -> ...
    // 之后: ... -> prevPrev -> curr -> prev -> curr->next -> ...

    if (prevPrev != nullptr) {
        prevPrev->next = curr;  // prevPrev 现在指向 curr
    } else {
        head = curr;            // prev 就是 head 的情况
    }

    prev->next = curr->next;    // prev 跳过 curr，指向 curr 的下一个
    curr->next = prev;          // curr 指向 prev，完成交换

    return head;
}

如果只关心值序列，也可以直接交换数据：

In [ ]:
// 简洁版：直接交换数据（只改变值的顺序，不改变节点位置）

In [ ]:
Node* transposeSimple(Node* head, int key) {
    if (head == nullptr || head->data == key) {
        return head;
    }

    Node* prev = nullptr;
    Node* curr = head;

    while (curr != nullptr && curr->data != key) {
        prev = curr;
        curr = curr->next;
    }

    if (curr == nullptr || prev == nullptr) {
        return head;
    }

    // 交换 curr 和 prev 的数据
    int temp = prev->data;
    prev->data = curr->data;
    curr->data = temp;

    return head;
}

注意：交换数据版本在数据域很大时效率不高（拷贝开销大）；如果外部保存了节点指针，或节点有独立身份，交换数据和交换节点位置也不是同一语义。下面的图解用指针交换版本来展示真实的位置变化。

**逐步图解：**

假设链表 `10 -> 20 -> 30 -> 40 -> NULL`，搜索 key = 30：

```text
搜索命中后:
  head -> [10] -> [20] -> [30] -> [40] -> nullptr

关键指针:
  prevPrev -> [10]
  prev     -> [20]
  curr     -> [30]

要把 curr 和 prev 交换，需要同时保存 prevPrev。
如果 prev 本来就是 head，那么 prevPrev 为 NULL，交换后 head 要改成 curr。
```

搜索到 30 后，将 `[30]` 和 `[20]` 交换位置：

```text
指针交换前：
  ... -> [10] -> [20] -> [30] -> [40] -> ...
        prevPrev  prev    curr

指针交换三步：
1. prevPrev->next = curr
2. prev->next = curr->next
3. curr->next = prev

指针交换后：
  head -> [10] -> [30] -> [20] -> [40] -> nullptr

含义:
  [30] 只向前移动一步，没有跳到头部。
```

如果接着再次搜索 30：

```text
当前链表：
  head -> [10] -> [30] -> [20] -> [40] -> nullptr

这次 prev 就是 head，prevPrev = NULL。
交换后需要执行 head = curr：

交换后:
  head -> [30] -> [10] -> [20] -> [40] -> nullptr

含义:
  经过两次搜索，[30] 才到头部。
```

对比移至头部法——搜索一次 30 就直接跳到头部。转置法需多次搜索才能逐步靠近头部。

**复杂度分析：**
- 时间复杂度：O(n)，搜索需要遍历
- 空间复杂度：O(1)
- **与移至头部法的对比**：转置法更适合元素访问频率分布较均匀的场景，避免偶尔搜索一次的冷数据把热数据挤到后面。

指针交换版本可能改变头节点，调用时同样要接住返回值：

In [ ]:
head = transpose(head, key);

---

**本章小结：**

| 操作 | 迭代版 | 递归版 | 理解关键 |
| ---- | ------ | ------ | -------- |
| 正序显示 | O(n) / O(1) | O(n) / O(n) | `p` 一路后移 |
| 反序显示 | 本节未给出 | O(n) / O(n) | 先到底，再回溯 |
| 计算节点数 | O(n) / O(1) | O(n) / O(n) | 访问一个加一 |
| 求和 | O(n) / O(1) | O(n) / O(n) | 累加每个 `data` |
| 求最大值 | O(n) / O(1) | O(n) / O(n) | 保存当前最大值 |
| 线性搜索 | O(n) / O(1) | O(n) / O(n) | 命中就返回节点 |
| 移至头部搜索 | O(n) / O(1) | 无 | 命中节点直接头插 |
| 转置搜索 | O(n) / O(1) | 无 | 命中节点前移一位 |

本章的显示、统计、查找等操作通常需要遍历链表，因此时间复杂度为 O(n)。但头插、已知前驱后的插入、给定节点的双向链表删除等操作可以是 O(1)。迭代版空间复杂度 O(1)，递归版 O(n)。实际项目中，若链表很长，优先使用迭代版以避免栈溢出。

**自测边界用例：**

| 操作 | 建议测试 |
| ---- | -------- |
| 遍历/统计 | 空链表、单节点链表、多节点链表 |
| 搜索 | 目标在头部、尾部、不存在、重复值只返回第一个 |
| 移至头部/转置 | 命中头节点、命中第二个节点、命中尾节点、未命中 |

> 单链表核心操作部分开始。

## 第三阶段：单链表核心操作

掌握了链表的基本结构和遍历方法后，接下来学习最核心的操作：**插入、删除和反转**。这些操作是几乎所有链表问题的基础。

---

### 7. 链表中的插入

插入是链表最常用的操作之一。与数组不同，链表插入无需移动元素，只需修改指针。

---

#### 7.1 在指定位置插入节点（Insert at Position）

##### 概念解释

"在指定位置插入"是指：给定一个位置编号 `pos`（从 0 开始计数），将新节点插入到该位置。
插入后，新节点成为链表中第 `pos` 个节点（0-indexed），原来该位置及之后的节点全部后移一位。
合法插入范围是 `[0, length]`：`pos == length` 表示尾部插入。

例如，链表 `1 -> 3 -> 5`，在 `pos = 1` 处插入 `2`，结果变为 `1 -> 2 -> 3 -> 5`。

##### 思路分析

分两种情况讨论：

**情况一：pos == 0，即在头部插入**

直接让新节点的 `next` 指向当前的 `head`，然后将 `head` 更新为新节点。

```text
改前:
  head → [1] → [3] → [5] → nullptr
  newNode → [2] → nullptr

改后:
  newNode->next = head
  head = newNode
  head → [2] → [1] → [3] → [5] → nullptr

含义:
  [2] 先接住旧 head，再成为新 head。
```

注意：这种情况会**改变 head 指针本身**，所以函数需要返回新的 `head`。

**情况二：pos > 0，在中间或尾部插入**

需要先找到位置 `pos` 的**前驱节点**（即第 `pos - 1` 个节点），记为 `prev`。然后：

1. 新节点的 `next` 指向 `prev->next`（先连上后面的链）
2. `prev->next` 指向新节点（再接入前面的链）

```text
改前:
  head → [1] → [3] → [5] → nullptr
  prev=[1], newNode=[2]

保存后继:
  newNode->next = prev->next
  [2] → [3] → [5] → nullptr

改后:
  prev->next = newNode
  head → [1] → [2] → [3] → [5] → nullptr

含义:
  先接住 [3]，再让 [1] 指向 [2]。
```

##### 指针操作顺序——先保存后继，再改前驱

这是初学者最容易犯的错误。插入操作有严格的顺序要求：

**正确顺序（先让新节点接住后半段，再让前驱指向新节点）：**
```text
改前:
  head → [1] → [3] → [5] → nullptr
  prev=[1], newNode=[2]

保存后继:
  newNode->next = prev->next
  [2] → [3] → [5] → nullptr

改后:
  prev->next = newNode
  head → [1] → [2] → [3] → [5] → nullptr

含义:
  不能先改 prev->next，否则 [3] 会失去入口。
```

**错误顺序（先改前驱）——反例演示：**

假设我们先执行 `prev->next = new_node`，再执行 `new_node->next = prev->next`：

```text
改前:
  head → [1] → [3] → [5] → nullptr
  prev=[1], newNode=[2]
错误改链:
  prev->next = newNode
  head → [1] → [2] → nullptr
  [3] → [5] → nullptr   （已失去入口）
继续错误:
  newNode->next = prev->next
  [2] → [2] → [2] → ...
含义:
  没有先保存 [3]，会丢链并形成自环。
```

而原来的节点 [3]、[5] 全部丢失，无法访问。

所以记住口诀：**先让新节点接住后半段，再把前驱接到新节点**。

##### 完整代码

In [ ]:
// 在链表的第 pos 个位置（0-indexed）插入值为 val 的节点
// 返回链表的头指针（因为 pos == 0 时 head 会改变）

In [ ]:
Node* insertAtPosition(Node* head, int pos, int val) {
    if (pos < 0) {
        std::cout << "位置不能为负数" << std::endl;
        return head;
    }

    Node* newNode = new Node(val);

    // 特殊情况：在头部插入（pos == 0）
    if (pos == 0) {
        newNode->next = head;
        return newNode;   // 新节点成为新的 head
    }

    // 一般情况：找到第 pos-1 个节点（前驱节点）
    Node* prev = head;
    for (int i = 0; i < pos - 1; i++) {
        if (prev == nullptr) {
            // pos 超出链表长度，插入失败
            std::cout << "位置 " << pos << " 超出链表范围" << std::endl;
            delete newNode;
            return head;
        }
        prev = prev->next;
    }

    // 检查 prev 是否为空（pos 刚好比长度大 1 的情况）
    if (prev == nullptr) {
        std::cout << "位置 " << pos << " 超出链表范围" << std::endl;
        delete newNode;
        return head;
    }

    // 核心操作：先让新节点接住后半段，再改前驱指向
    newNode->next = prev->next;   // 步骤 1：新节点连上后面的链
    prev->next = newNode;          // 步骤 2：前驱连上新节点

    return head;
}

##### 逐步图解

以链表 `10 -> 20 -> 30 -> 40` 为例，在 `pos = 2` 处插入 `25`：

```text
改前:
  head → [10] → [20] → [30] → [40] → nullptr
  pos=2, prev=[20], newNode=[25]

保存后继:
  newNode->next = prev->next
  [25] → [30] → [40] → nullptr

改后:
  prev->next = newNode
  head → [10] → [20] → [25] → [30] → [40] → nullptr

含义:
  [25] 先接住 [30]，再接到 [20] 后面。
```

##### 复杂度分析

| 指标 | 复杂度 | 说明 |
| ---- | ------ | ---- |
| 时间 | O(n) | 最坏情况需要遍历到第 pos-1 个节点 |
| 空间 | O(1) | 只创建了一个新节点 |

特别说明：如果在头部插入（pos = 0），时间复杂度为 O(1)，因为不需要遍历。

---

#### 7.2 使用头插法创建链表

##### 概念解释

头插法（Head Insertion）是一种创建链表的方法：每次都将新节点插入到链表的**头部**。
由于总是插在最前面，最终得到的链表顺序与输入顺序**相反**。

##### 思路分析

```text
初始状态:
  输入顺序: 1, 2, 3, 4
  head → nullptr

每次头插:
  Push(1): head → [1] → nullptr
  Push(2): head → [2] → [1] → nullptr
  Push(3): head → [3] → [2] → [1] → nullptr
  Push(4): head → [4] → [3] → [2] → [1] → nullptr

含义:
  新节点总在 head 前，结果与输入顺序相反。
```

每一步的操作：
1. 创建新节点 `newNode`
2. `newNode->next = head`（新节点指向原来的头）
3. `head = newNode`（更新 head 为新节点）

##### 完整代码

In [ ]:
// 使用头插法，从数组创建链表
// 返回链表的头指针

In [ ]:
Node* createListByHeadInsertion(int arr[], int n) {
    Node* head = nullptr;

    for (int i = 0; i < n; i++) {
        Node* newNode = new Node(arr[i]);
        newNode->next = head;   // 新节点指向原来的头
        head = newNode;          // 更新 head
    }

    return head;
}

##### 逐步图解

以数组 `[10, 20, 30]` 为例：

```text
初始状态:
  head → nullptr
插入 10:
  newNode->next = head
  head → [10] → nullptr
插入 20:
  newNode->next = head
  head → [20] → [10] → nullptr
插入 30:
  head → [30] → [20] → [10] → nullptr
含义:
  先接旧 head，再更新 head。
```

##### 复杂度分析

| 指标 | 复杂度 | 说明 |
| ---- | ------ | ---- |
| 时间 | O(n) | n 个元素，每个插入 O(1) |
| 空间 | O(n) | 创建了 n 个节点 |

---

#### 7.3 使用尾插法创建链表

##### 概念解释

尾插法（Tail Insertion）是一种创建链表的方法：每次都将新节点插入到链表的**尾部**。
最终得到的链表顺序与输入顺序**一致**。

与头插法相比，尾插法需要额外维护一个 `tail` 指针，指向链表的最后一个节点，以避免每次插入都从头遍历到尾部。

##### 思路分析

```text
初始状态:
  输入顺序: 1, 2, 3, 4
  head=nullptr, tail=nullptr

每次尾插:
  Append(1): head/tail → [1] → nullptr
  Append(2): head → [1] → [2] → nullptr, tail=[2]
  Append(3): head → [1] → [2] → [3] → nullptr, tail=[3]
  Append(4): head → [1] → [2] → [3] → [4] → nullptr, tail=[4]

含义:
  新节点接在 tail 后，tail 再移动到新尾节点。
```

每一步的操作：
1. 创建新节点 `newNode`
2. 如果链表为空，`head = tail = newNode`
3. 否则，`tail->next = newNode`，然后 `tail = newNode`

##### 完整代码

In [ ]:
// 使用尾插法，从数组创建链表
// 返回链表的头指针

In [ ]:
Node* createListByTailInsertion(int arr[], int n) {
    if (n <= 0) return nullptr;

    Node* head = new Node(arr[0]);
    Node* tail = head;

    for (int i = 1; i < n; i++) {
        Node* newNode = new Node(arr[i]);
        tail->next = newNode;   // 尾节点指向新节点
        tail = newNode;          // 更新 tail
    }

    return head;
}

##### 逐步图解

以数组 `[10, 20, 30]` 为例：

```text
初始状态:
  head=nullptr, tail=nullptr
插入 10:
  head/tail → [10] → nullptr
插入 20:
  改前: head → [10] → nullptr, tail=[10], newNode=[20]
  改链: tail->next = newNode
  改后: head → [10] → [20] → nullptr, tail=[20]
插入 30:
  head → [10] → [20] → [30] → nullptr, tail=[30]
含义:
  先接到旧尾后，再更新 tail。
```

##### 头插法与尾插法对比

| 对比项 | 头插法 | 尾插法 |
| ------ | ------ | ------ |
| 新节点位置 | 放到 `head` 前 | 接到 `tail` 后 |
| `head` 变化 | 每次都变 | 只在空表时变 |
| `tail` 变化 | 通常不维护 | 每次都更新 |
| 输出顺序 | 与输入相反 | 与输入相同 |
| 单次插入 | O(1) | O(1) |
| 适合场景 | 栈、逆序构建 | 保持输入顺序 |

---

#### 7.4 在有序链表中插入

##### 概念解释

假设链表已经按升序排列，要求插入一个新节点后，链表仍然保持升序。
需要找到**第一个大于或等于待插入值**的节点，将新节点插入到它前面。

##### 思路分析

根据插入位置不同，有三种情况：

**情况一：插入到头部（新值比所有元素都小）**
```text
改前:
  val=0
  head → [3] → [5] → [8] → nullptr

改指针后:
  newNode->next = head
  head → [0] → [3] → [5] → [8] → nullptr

含义:
  0 不大于头节点 3，直接按头插处理。
```

**情况二：插入到中间**
```text
查找位置:
  val=6
  head → [3] → [5] → [8] → nullptr
  curr=[5], curr->next=[8]

含义:
  curr->data < 6；
  curr->next->data >= 6；
  所以 [6] 插在 curr 和 curr->next 之间。
```

**情况三：插入到尾部（新值比所有元素都大）**
```text
改前:
  val=10
  head → [3] → [5] → [8] → nullptr
  curr=[8], curr->next=nullptr

最终状态:
  head → [3] → [5] → [8] → [10] → nullptr

含义:
  curr->next == nullptr；
  新节点接到尾节点 [8] 后面。
```

##### 完整代码

In [ ]:
// 在有序（升序）链表中插入值为 val 的节点
// 返回链表的头指针

In [ ]:
Node* insertInSortedList(Node* head, int val) {
    Node* newNode = new Node(val);

    // 情况一：链表为空，或新值比头节点小，插到头部
    if (head == nullptr || val <= head->data) {
        newNode->next = head;
        return newNode;
    }

    // 情况二和三：找到合适的插入位置
    // curr 从 head 开始，找第一个 data >= val 的节点的前驱
    Node* curr = head;
    while (curr->next != nullptr && curr->next->data < val) {
        curr = curr->next;
    }

    // 此时 curr->next 要么是 nullptr（情况三），要么 data >= val（情况二）
    newNode->next = curr->next;
    curr->next = newNode;

    return head;
}

##### 图解（情况二：中间插入）

以链表 `2 -> 5 -> 8 -> 12` 插入值 `7` 为例：

```text
改前:
  head → [2] → [5] → [8] → [12] → nullptr
  curr=[5], newNode=[7]
找到位置:
  curr->next=[8]，因为 5 < 7 <= 8。
保存后继:
  newNode->next = curr->next
  [7] → [8] → [12] → nullptr
改指针后:
  curr->next = newNode
  head → [2] → [5] → [7] → [8] → [12] → nullptr
含义:
  先让新节点接住后继，再让 curr 指向新节点。
```

##### 复杂度分析

| 指标 | 复杂度 | 说明 |
| ---- | ------ | ---- |
| 时间 | O(n) | 最坏情况遍历整个链表（插入到尾部） |
| 空间 | O(1) | 只创建了一个新节点 |

---

### 8. 链表中的删除

删除是插入的逆操作。链表删除只需修改指针，但要特别注意**内存释放**。

---

#### 8.1 删除指定位置的节点

##### 概念解释

给定位置 `pos`（0-indexed），删除链表中第 `pos` 个节点，并返回被删除节点的值。

##### 思路分析

**情况一：删除头节点（pos == 0）**

保存头节点的值，将 `head` 指向 `head->next`，然后释放原头节点。

```text
改前:
  head → [1] → [3] → [5] → nullptr

保存后继:
  temp=head, val=1
  head = head->next

改后:
  head → [3] → [5] → nullptr
  delete temp   // 释放原 [1]
```

**情况二：删除中间或尾部节点（pos > 0）**

找到被删节点的**前驱节点** `prev`，然后：
1. `temp = prev->next`（临时指针保存要删除的节点）
2. `prev->next = temp->next`（跳过被删节点）
3. `delete temp`（释放内存）

```text
改前:
  head → [1] → [3] → [5] → nullptr
  prev=[1]

保存后继:
  temp = prev->next
  temp=[3], temp->next=[5]

改后:
  prev->next = temp->next
  head → [1] → [5] → nullptr
  delete temp   // 释放 [3]
```

##### 完整代码

In [ ]:
// 删除链表中第 pos 个节点（0-indexed）
// 返回被删除节点的值；如果 pos 非法，返回 -1 并打印提示
// 注意：head 可能被修改，所以参数用引用

In [ ]:
int deleteAtPosition(Node*& head, int pos) {
    if (pos < 0) {
        std::cout << "位置不能为负数" << std::endl;
        return -1;
    }

    if (head == nullptr) {
        std::cout << "链表为空，无法删除" << std::endl;
        return -1;
    }

    // 情况一：删除头节点
    if (pos == 0) {
        Node* temp = head;
        int val = temp->data;
        head = head->next;
        delete temp;
        return val;
    }

    // 情况二：找到第 pos-1 个节点（前驱）
    Node* prev = head;
    for (int i = 0; i < pos - 1; i++) {
        if (prev->next == nullptr) {
            std::cout << "位置 " << pos << " 超出链表范围" << std::endl;
            return -1;
        }
        prev = prev->next;
    }

    // 检查要删除的节点是否存在
    if (prev->next == nullptr) {
        std::cout << "位置 " << pos << " 超出链表范围" << std::endl;
        return -1;
    }

    // 核心操作
    Node* temp = prev->next;         // 临时保存要删除的节点
    int val = temp->data;            // 保存返回值
    prev->next = temp->next;         // 跳过被删节点
    delete temp;                     // 释放内存

    return val;
}

这里用 `-1` 表示删除失败是为了简化示例；如果节点值本身可能为 `-1`，工程代码应改用 `std::optional<int>`、`bool + 输出参数` 或异常来区分失败和真实数据。

##### 逐步图解

以链表 `10 -> 20 -> 30 -> 40` 删除 `pos = 2`（值为 30 的节点）为例：

```text
改前:
  head → [10] → [20] → [30] → [40] → nullptr
  pos=2, prev=[20]
保存后继:
  temp = prev->next
  temp=[30], temp->next=[40]
改链:
  prev->next = temp->next
  head → [10] → [20] → [40] → nullptr
释放:
  delete temp
  被释放节点: [30]
```

##### 复杂度分析

| 指标 | 复杂度 | 说明 |
| ---- | ------ | ---- |
| 时间 | O(n) | 最坏情况遍历到第 pos-1 个节点 |
| 空间 | O(1) | 只用了临时指针 |

---

#### 8.2 按值删除节点

##### 概念解释

给定一个目标值 `target`，删除链表中**第一个**值等于 `target` 的节点。

##### 思路分析

需要处理两个特殊情况：
1. 要删除的节点恰好是头节点
2. 要删除的节点不存在

其余情况与按位置删除类似：找到前驱，跳过被删节点，释放内存。

##### 完整代码

In [ ]:
// 删除链表中第一个值等于 target 的节点
// 返回 true 表示删除成功，false 表示未找到

In [ ]:
bool deleteByValue(Node*& head, int target) {
    if (head == nullptr) return false;

    // 特殊情况：头节点就是要删除的
    if (head->data == target) {
        Node* temp = head;
        head = head->next;
        delete temp;
        return true;
    }

    // 找前驱节点
    Node* prev = head;
    while (prev->next != nullptr && prev->next->data != target) {
        prev = prev->next;
    }

    // 没找到
    if (prev->next == nullptr) return false;

    // 找到了，执行删除
    Node* temp = prev->next;
    prev->next = temp->next;
    delete temp;
    return true;
}

##### 复杂度分析

| 指标 | 复杂度 | 说明 |
| ---- | ------ | ---- |
| 时间 | O(n) | 最坏情况遍历整个链表 |
| 空间 | O(1) | 只用了临时指针 |

---

### 9. 链表的有序性操作

有序链表是链表的一种常见状态，很多算法都依赖于链表的有序性。

---

#### 9.1 检查链表是否有序

##### 概念解释

判断链表是否按**升序（非降序）**排列。空链表和只有一个节点的链表视为有序。

##### 思路分析

从头到尾遍历，比较每一对相邻节点。如果发现某一对 `curr->data > curr->next->data`，则链表无序。

##### 完整代码

In [ ]:
// 检查链表是否按升序（非降序）排列

In [ ]:
bool isSorted(Node* head) {
    // 空链表或单个节点，视为有序
    if (head == nullptr || head->next == nullptr) {
        return true;
    }

    Node* curr = head;
    while (curr->next != nullptr) {
        if (curr->data > curr->next->data) {
            return false;   // 发现逆序对
        }
        curr = curr->next;
    }

    return true;
}

##### 复杂度分析

| 指标 | 复杂度 | 说明 |
| ---- | ------ | ---- |
| 时间 | O(n) | 最坏情况遍历一次 |
| 空间 | O(1) | 只用了一个指针 |

---

#### 9.2 移除有序链表中的重复元素

##### 概念解释

给定一个**已排序**的链表，删除其中所有重复的元素，使得每个值只出现一次。

例如：`2 -> 3 -> 3 -> 5 -> 5 -> 5 -> 8` 变为 `2 -> 3 -> 5 -> 8`。

由于链表已排序，重复的元素一定是**相邻**的，这大大简化了问题。

##### 思路分析

遍历链表，对每个节点检查它的下一个节点是否具有相同的值：
- 如果相同，删除下一个节点（注意释放内存）
- 如果不同，移到下一个节点

关键点：删除一个重复节点后，**不要**立即前移，因为可能还有更多重复。只有当 `curr->next` 的值不同时，才移动 `curr`。

##### 完整代码

In [ ]:
// 移除有序链表中的重复元素（每个值只保留一个）

In [ ]:
Node* removeDuplicates(Node* head) {
    if (head == nullptr || head->next == nullptr) {
        return head;
    }

    Node* curr = head;
    while (curr->next != nullptr) {
        if (curr->data == curr->next->data) {
            // 发现重复，删除 curr->next
            Node* temp = curr->next;
            curr->next = temp->next;
            delete temp;
            // 注意：curr 不移动，继续检查新的 curr->next
        } else {
            // 不重复，移到下一个节点
            curr = curr->next;
        }
    }

    return head;
}

##### 逐步图解

以链表 `2 -> 3 -> 3 -> 5 -> 5 -> 5 -> 8` 为例：

```text
初始状态:
  head → [2] → [3] → [3] → [5] → [5] → [5] → [8] → nullptr
遇到重复 3:
  curr=[3], temp=curr->next=[3]
  curr->next = temp->next
  head → [2] → [3] → [5] → [5] → [5] → [8] → nullptr
  delete temp
遇到重复 5:
  第一次删除后: head → [2] → [3] → [5] → [5] → [8] → nullptr
  第二次删除后: head → [2] → [3] → [5] → [8] → nullptr
含义:
  删除重复后 curr 不动，继续检查新的 curr->next。
```

##### 复杂度分析

| 指标 | 复杂度 | 说明 |
| ---- | ------ | ---- |
| 时间 | O(n) | 循环次数与节点数线性相关，每轮要么删除一个节点，要么前进一步 |
| 空间 | O(1) | 只用了临时指针 |

---

### 10. 链表反转（Reverse）—— 三种方法

链表反转是链表操作中最经典、最重要、也是面试中出现频率最高的题目之一。
本节介绍三种不同的反转方法，从易到难，各有优缺点。

反转前后的效果：
```text
反转前：head -> [1] -> [2] -> [3] -> [4] -> [5] -> nullptr

反转后：head -> [5] -> [4] -> [3] -> [2] -> [1] -> nullptr
        （head 指向原来的尾节点）
```

---

#### 10.1 方法一：使用辅助数组反转节点值

##### 概念解释

最直观的方法：把链表的所有值复制到一个数组中，然后从数组末尾开始，反向写回链表。

##### 思路分析

1. 遍历链表，将每个节点的值存入数组
2. 再次遍历链表，从数组的最后一个元素开始，依次写回每个节点

这种方法不改变节点之间的连接关系，只改变节点中存储的值；因此它不是指针意义上的“链表反转”，返回的仍是原来的 `head`。

##### 完整代码

In [ ]:
#include <vector>

In [ ]:
struct Node {
    int data;
    Node* next;
    Node(int val) : data(val), next(nullptr) {}
};

In [ ]:
// 方法一：使用辅助数组反转链表
Node* reverseUsingArray(Node* head) {
    if (head == nullptr || head->next == nullptr) {
        return head;
    }

    // 第一步：将所有值存入数组
    std::vector<int> values;
    Node* curr = head;
    while (curr != nullptr) {
        values.push_back(curr->data);
        curr = curr->next;
    }

    // 第二步：从数组末尾反向写回链表
    curr = head;
    auto i = values.size();
    while (curr != nullptr) {
        curr->data = values[--i];
        curr = curr->next;
    }

    return head;
}

##### 图解

```text
初始状态:
  head → [1] → [2] → [3] → [4] → [5] → nullptr

保存到数组:
  values = [1, 2, 3, 4, 5]

反向写回:
  写回 5: head → [5] → [2] → [3] → [4] → [5] → nullptr
  写回 4: head → [5] → [4] → [3] → [4] → [5] → nullptr
  ...
  写回 1: head → [5] → [4] → [3] → [2] → [1] → nullptr

含义:
  节点连接不变，只交换 data 的值。
```

##### 复杂度分析

| 指标 | 复杂度 | 说明 |
| ---- | ------ | ---- |
| 时间 | O(n) | 遍历两次链表 |
| 空间 | O(n) | 额外的数组空间 |

---

#### 10.2 方法二：滑动指针法（三指针迭代反转）

##### 概念解释

这是最常用、最高效的迭代方法。使用三个指针在链表上"滑动"，逐步将每个节点的 `next` 指针反转方向。

三个指针分别是：
- `prev`：已经反转的部分的头（初始为 `nullptr`）
- `curr`：当前正在处理的节点（初始为 `head`）
- `next_node`：保存 `curr` 的下一个节点（防止断链后丢失）

有些资料会把三者写成 `r / q / p`，本节对应关系是：`r = prev`，`q = curr`，`p = next_node`。

##### 思路分析

每一步的核心操作：
1. 保存 `curr->next` 到 `next_node`（防止丢失后面的节点）
2. 将 `curr->next` 指向 `prev`（反转指向）
3. 移动 `prev = curr`
4. 移动 `curr = next_node`

循环直到 `curr == nullptr`，此时 `prev` 就是新的头节点。

##### 完整代码

In [ ]:
// 方法二：三指针迭代反转

In [ ]:
Node* reverseIterative(Node* head) {
    Node* prev = nullptr;
    Node* curr = head;

    while (curr != nullptr) {
        Node* next_node = curr->next;  // 保存下一个节点
        curr->next = prev;             // 反转指向
        prev = curr;                   // prev 前移
        curr = next_node;              // curr 前移
    }

    // 循环结束时，prev 指向原链表的最后一个节点，即新的头
    return prev;
}

调用时必须更新入口指针：

In [ ]:
head = reverseIterative(head);

##### 逐步图解

以链表 `1 -> 2 -> 3 -> 4 -> 5 -> nullptr` 为例，抓住前两轮就能看清模式：

```text
初始状态:
  prev=nullptr, curr=[1]
  剩余链: [1] → [2] → [3] → [4] → [5] → nullptr
第 1 轮:
  next_node=[2]
  改链后: [1] → nullptr
  移动后: prev=[1], curr=[2]
第 2 轮:
  next_node=[3]
  改链后: [2] → [1] → nullptr
  移动后: prev=[2], curr=[3]
继续到结束:
  prev=[5], curr=nullptr
  新 head → [5] → [4] → [3] → [2] → [1] → nullptr
```

##### 指针变化汇总表

表中“保存后继”是反转指针之前的状态，“轮末状态”是执行完 `prev = curr; curr = next_node;` 之后的状态。

| 轮次 | 先保存 `next` | 改 `curr->next` | 轮末状态 |
| ---- | --------------- | --------------- | -------- |
| 初始 | - | - | `prev=nullptr`，`curr=[1]` |
| 1 | `[2]` | `[1] -> nullptr` | `prev=[1]`，`curr=[2]` |
| 2 | `[3]` | `[2] -> [1]` | `prev=[2]`，`curr=[3]` |
| 3 | `[4]` | `[3] -> [2]` | `prev=[3]`，`curr=[4]` |
| 4 | `[5]` | `[4] -> [3]` | `prev=[4]`，`curr=[5]` |
| 5 | `nullptr` | `[5] -> [4]` | `prev=[5]`，`curr=nullptr` |

##### 复杂度分析

| 指标 | 复杂度 | 说明 |
| ---- | ------ | ---- |
| 时间 | O(n) | 遍历一次链表 |
| 空间 | O(1) | 只用了三个指针 |

这是**最优解法**，面试中优先使用。

---

#### 10.3 方法三：递归反转

##### 概念解释

递归法利用函数调用栈，先递归到链表尾部，然后在回溯过程中逐个反转指针。

核心思想：如果链表后面的部分已经反转好了，那么只需要把当前节点接到已反转部分的末尾。

##### 递归的核心逻辑

假设链表是 `1 -> 2 -> 3 -> 4 -> 5`：

```text
递归前:
  head → [1] → [2] → [3] → [4] → [5] → nullptr

假设后半段已经反转:
  newHead → [5] → [4] → [3] → [2] → nullptr
  head=[1], head->next=[2]

回溯改链:
  head->next->next = head   // [2] → [1]
  head->next = nullptr      // [1] 成为尾节点
```

对应的两行关键代码：

In [ ]:
head->next->next = head;
// 让已反转部分的尾（原 head->next）指向 head
head->next = nullptr;
// head 现在是尾节点，next 置空

##### 完整代码

In [ ]:
// 方法三：递归反转

In [ ]:
Node* reverseRecursive(Node* head) {
    // 基础情况：空链表或只有一个节点
    if (head == nullptr || head->next == nullptr) {
        return head;
    }

    // 递归调用：反转 head 之后的部分
    Node* newHead = reverseRecursive(head->next);

    // 回溯时反转指针
    head->next->next = head;   // 让后继节点指回自己
    head->next = nullptr;       // 自己的 next 置空

    return newHead;   // newHead 始终是原链表的最后一个节点
}

##### 递归展开图

以链表 `1 -> 2 -> 3 -> 4 -> 5` 为例，完整展示递归调用栈和回溯过程：

```text
递归深入:
  reverse(1)
    reverse(2)
      reverse(3)
        reverse(4)
          reverse(5)  // 返回 [5]
回溯改链:
  head=[4]: [5] → [4] → nullptr
  head=[3]: [5] → [4] → [3] → nullptr
  head=[2]: [5] → [4] → [3] → [2] → nullptr
  head=[1]: [5] → [4] → [3] → [2] → [1] → nullptr
含义:
  每层先让后继指回当前，再把当前 next 置空。
```

##### 递归调用栈总结

```text
调用方向（深入）:
  reverse(1..5)
    reverse(2..5)
      reverse(3..5)
        reverse(4..5)
          reverse(5)  // 基础情况

返回方向（回溯）:
  每一层都返回同一个 newHead=[5]
```

每层回溯时都先执行 `head->next->next = head`，再执行 `head->next = nullptr`，及时断开旧方向，避免最终留下环。

| 回溯层 | 当前 `head` | 关键操作 | 已反转前缀 |
| ------ | ----------- | -------- | ---------- |
| 第五层 | `[5]` | 基础情况 | `[5]` |
| 第四层 | `[4]` | `[5]` 指回 `[4]` | `[5] -> [4]` |
| 第三层 | `[3]` | `[4]` 指回 `[3]` | `[5] -> [4] -> [3]` |
| 第二层 | `[2]` | `[3]` 指回 `[2]` | `[5] -> [4] -> [3] -> [2]` |
| 第一层 | `[1]` | `[2]` 指回 `[1]` | `[5] -> [4] -> [3] -> [2] -> [1]` |

##### 复杂度分析

| 指标 | 复杂度 | 说明 |
| ---- | ------ | ---- |
| 时间 | O(n) | 递归 n 层，每层 O(1) |
| 空间 | O(n) | 递归调用栈深度为 n |

递归法在链表很长时可能导致**栈溢出**（stack overflow），因为调用栈深度等于链表长度。

---

#### 10.4 三种反转方法的对比总结

| 方法 | 核心思路 | 时间 | 空间 | 推荐场景 |
| ---- | -------- | ---- | ---- | -------- |
| 辅助数组法 | 把值存入数组再写回 | O(n) | O(n) | 理解过渡 |
| 三指针迭代法 | 原地改 `next` 方向 | O(n) | O(1) | 首选写法 |
| 递归法 | 先到底，再回溯改指针 | O(n) | O(n) | 理解递归 |

**建议**：
- 面试和工程代码优先写**三指针迭代法**，它真正反转节点连接，空间也最省
- 如果面试官要求用递归，写出递归版本并说明栈溢出的风险
- 辅助数组法可以用来验证其他方法的正确性，但一般不作为面试答案

**核心操作边界用例：**

| 操作 | 必测场景 |
| ---- | -------- |
| 插入 | `pos = 0`、`pos = length`、负数、超过 `length` |
| 删除 | 空链表、删除头节点、删除尾节点、删除不存在的位置或值 |
| 去重 | 空链表、单节点、全重复、无重复、未排序输入（应先说明前提） |
| 反转 | 空链表、单节点、两个节点、多个节点 |

## 第四阶段：单链表高级操作

在前三阶段中，我们学会了单链表的基本操作、查找统计、以及有序性整理。本阶段将进入更复杂但极其常见的操作：连接、合并、环检测，最后用 C++ 类将所有操作封装。

---

### 11. 连接两个链表（Concatenation）

#### 11.1 概念解释

"连接"很简单：把链表 B 的所有节点接到链表 A 的末尾。

```text
连接前:

  headA → [10] → [20] → [30] → nullptr
  headB → [40] → [50] → nullptr

连接后:

  headA → [10] → [20] → [30] → [40] → [50] → nullptr
```

连接的关键点：

- 只需把 A 的尾节点的 `next` 指向 B 的头节点
- 无需复制或移动任何节点
- 连接后 A 和 B 共享节点，这是重要的内存管理注意事项
- 前提：两条链表应当无环且互不相交，不能把同一条链表连接到自己后面

#### 11.2 思路分析

| 情况 | 操作 | 返回 |
| ---- | ---- | ---- |
| A 为空 | 不需要连接 | `headB` |
| B 为空 | 不需要连接 | `headA` |
| 都非空 | 找到 A 的尾节点 | 继续 |
| 找到尾节点后 | `tail->next = headB` | `headA` |

#### 11.3 代码实现

In [ ]:
#include <iostream>
using namespace std;

In [ ]:
struct Node {
    int data;
    Node* next;
    Node(int val) : data(val), next(nullptr) {}
};

In [ ]:
// 连接两个链表：将 B 接在 A 末尾，返回 A 的 head
Node* concatenate(Node* headA, Node* headB) {
    // 情况 1: A 为空，直接返回 B
    if (headA == nullptr) return headB;

    // 情况 2: B 为空，直接返回 A
    if (headB == nullptr) return headA;

    // 情况 3: 两个都不为空，找到 A 的尾节点
    Node* tail = headA;
    while (tail->next != nullptr) {
        tail = tail->next;
    }

    // 将 A 的尾节点指向 B 的头节点
    tail->next = headB;
    return headA;
}

// 辅助函数：创建链表（尾插法）

In [ ]:
Node* createList(int arr[], int n) {
    if (n <= 0) return nullptr;
    Node* head = new Node(arr[0]);
    Node* tail = head;
    for (int i = 1; i < n; i++) {
        tail->next = new Node(arr[i]);
        tail = tail->next;
    }
    return head;
}

In [ ]:
void deleteList(Node* head) {
    while (head != nullptr) {
        Node* nextNode = head->next;
        delete head;
        head = nextNode;
    }
}

In [ ]:
void display(Node* head) {
    Node* temp = head;
    while (temp != nullptr) {
        cout << temp->data << " -> ";
        temp = temp->next;
    }
    cout << "NULL" << endl;
}

In [ ]:
int main() {
    int arrA[] = {10, 20, 30};
    int arrB[] = {40, 50};
    Node* headA = createList(arrA, 3);
    Node* headB = createList(arrB, 2);

    cout << "链表 A: ";  display(headA);
    cout << "链表 B: ";  display(headB);

    headA = concatenate(headA, headB);
    headB = nullptr;  // B 的节点已接到 A 后面，不能再作为独立链表释放
    cout << "连接后: ";  display(headA);

    deleteList(headA);
    return 0;
}

In [ ]:
main();

输出：

```text
链表 A: 10 -> 20 -> 30 -> NULL
链表 B: 40 -> 50 -> NULL
连接后: 10 -> 20 -> 30 -> 40 -> 50 -> NULL
```

#### 11.4 逐步图解

**初始状态：**

```text
链表 A:
  headA → [10] → [20] → [30] → nullptr

链表 B:
  headB → [40] → [50] → nullptr
```

**步骤 1-2: A 和 B 都不为空，进入主逻辑。**

**步骤 3: 从 headA 开始遍历，找到尾节点。tail 依次经过 10、20，最终停在 30（其 next 为 NULL）：**

```text
headA → [10] → [20] → [30] → nullptr
tail=[30], tail->next=nullptr
```

**步骤 4: 将 tail->next 指向 headB，连接完成：**

```text
连接前（两条独立链）:
  headA → [10] → [20] → [30] → nullptr
  headB → [40] → [50] → nullptr
  tail=[30], headB=[40]
只改一条边: tail->next = headB
连接后:
  headA → [10] → [20] → [30] → [40] → [50] → nullptr
  headB=[40]  // 仍是别名，不再单独释放
所有权转移:
  headB = nullptr
  headA → [10] → [20] → [30] → [40] → [50] → nullptr
  headB → nullptr
含义:
  [40] 和 [50] 已由 A 拥有；headB 不再删除这段链。
```

注意：裸指针版本的 `concatenate` 只能返回新的 `headA`，不会自动把调用者手里的 `headB` 改成 `nullptr`。如果连接后由 A 统一释放整条链，就应把 `headB` 视为已交出所有权，并在调用处清空它。

#### 11.5 内存管理注意事项

连接操作只修改了一个指针，未创建新节点。因此：

- **危险操作**：若连接后分别对 headA 和 headB 调用 delete，节点 40、50 会被释放两次，程序崩溃
- **正确做法**：连接后只通过 headA 管理整条链表，将 headB 视为"已交出所有权"

#### 11.6 复杂度分析

| 操作 | 时间复杂度 | 说明 |
| ---- | ---------- | ---- |
| 找 A 的尾节点 | O(n) | n 是链表 A 的长度 |
| 接上 B | O(1) | 一次 `tail->next` |
| 总计 | O(n) | 只遍历 A |

空间复杂度：O(1)。如果额外存储了尾节点指针，连接操作可以优化到 O(1)。

---

### 12. 合并两个有序链表（Merge Two Sorted Lists）

#### 12.1 概念解释

**合并**与**连接**不同：连接只把 B 接到 A 尾部；合并要求两链均已有序，结果也必须有序。

```text
合并前:
  p → [10] → [30] → [50] → nullptr
  q → [20] → [40] → [60] → nullptr

合并后:
  [10] → [20] → [30] → [40] → [50] → [60] → nullptr

每轮比较 p、q 当前节点，取较小者接到结果链尾，再移动对应指针。
```

合并是归并排序（Merge Sort）的核心步骤，也是链表算法中最重要的操作之一。

前提：参与合并的两条链表都应当有序、无环且互不相交。合并过程会重连原有节点，不会复制新节点。

#### 12.2 思路分析：双指针法

用两个指针 `p` 和 `q` 分别指向链表 A 和 B 的当前节点，每次比较 `p->data` 和 `q->data`，把较小的那个接到结果链表的尾部。

```text
核心循环逻辑:

while (p 和 q 都没遍历完) {
    if (p->data <= q->data) {
        把 p 接到结果末尾
        p 前进一步
    } else {
        把 q 接到结果末尾
        q 前进一步
    }
}
// 循环结束后，把剩余链表直接接上
```

#### 12.3 不带哨兵节点的实现

不带哨兵节点时，需要特殊处理第一个节点的选取。

In [ ]:
#include <iostream>
using namespace std;

In [ ]:
struct Node {
    int data;
    Node* next;
    Node(int val) : data(val), next(nullptr) {}
};

In [ ]:
// 合并两个有序链表（不带哨兵节点）
Node* mergeSorted(Node* headA, Node* headB) {
    if (headA == nullptr) return headB;
    if (headB == nullptr) return headA;

    Node* resultHead = nullptr;  // 结果链表的头
    Node* resultTail = nullptr;  // 结果链表的尾
    Node* p = headA;
    Node* q = headB;

    // 步骤 1: 确定结果链表的第一个节点
    if (p->data <= q->data) {
        resultHead = resultTail = p;
        p = p->next;
    } else {
        resultHead = resultTail = q;
        q = q->next;
    }

    // 步骤 2: 依次取较小的节点接到末尾
    while (p != nullptr && q != nullptr) {
        if (p->data <= q->data) {
            resultTail->next = p;
            resultTail = p;
            p = p->next;
        } else {
            resultTail->next = q;
            resultTail = q;
            q = q->next;
        }
    }

    // 步骤 3: 接上剩余部分
    resultTail->next = (p != nullptr) ? p : q;
    return resultHead;
}

// 辅助函数

In [ ]:
Node* createList(int arr[], int n) {
    if (n <= 0) return nullptr;
    Node* head = new Node(arr[0]);
    Node* tail = head;
    for (int i = 1; i < n; i++) {
        tail->next = new Node(arr[i]);
        tail = tail->next;
    }
    return head;
}

In [ ]:
void deleteList(Node* head) {
    while (head != nullptr) {
        Node* nextNode = head->next;
        delete head;
        head = nextNode;
    }
}

In [ ]:
void display(Node* head) {
    for (Node* t = head; t != nullptr; t = t->next)
        cout << t->data << " -> ";
    cout << "NULL" << endl;
}

In [ ]:
int main() {
    int arrA[] = {10, 30, 50, 70};
    int arrB[] = {20, 40, 60};
    Node* headA = createList(arrA, 4);
    Node* headB = createList(arrB, 3);

    cout << "链表 A: ";  display(headA);
    cout << "链表 B: ";  display(headB);

    Node* merged = mergeSorted(headA, headB);
    headA = headB = nullptr;  // merged 已接管两条链的节点，避免把 A/B 当独立链释放
    cout << "合并后: ";  display(merged);

    deleteList(merged);
    return 0;
}

In [ ]:
main();

输出：

```text
链表 A: 10 -> 30 -> 50 -> 70 -> NULL
链表 B: 20 -> 40 -> 60 -> NULL
合并后: 10 -> 20 -> 30 -> 40 -> 50 -> 60 -> 70 -> NULL
```

#### 12.4 逐步图解（不带哨兵节点版本）

以 A = {10, 30, 50}、B = {20, 40, 60} 为例。

**初始状态：**

```text
  p → [10] → [30] → [50] → nullptr
  q → [20] → [40] → [60] → nullptr
  resultHead = resultTail = nullptr
```

**选取第一个节点：10 < 20，选 [10]。**

```text
  resultHead → [10] → nullptr
  resultTail = [10]
  p → [30] → [50] → nullptr
  q → [20] → [40] → [60] → nullptr
```

**第 1 轮循环：比较 30 和 20，20 较小，将节点 20 接到结果末尾。**

```text
连接前:

  resultHead → [10] → nullptr
  resultTail=[10], q=[20]

连接后:

  resultHead → [10] → [20] → nullptr
  resultTail=[20]
  q → [40] → [60] → nullptr
```

**第 2 轮循环：比较 30 和 40，30 较小，将节点 30 接到结果末尾。**

```text
连接前:

  resultHead → [10] → [20] → nullptr
  resultTail=[20], p=[30]

连接后:

  resultHead → [10] → [20] → [30] → nullptr
  resultTail=[30]
  p → [50] → nullptr
```

**第 3 轮循环：比较 50 和 40，40 较小，将节点 40 接到结果末尾。**

```text
连接后:

  resultHead → [10] → [20] → [30] → [40] → nullptr
  resultTail=[40]
  p=[50], q=[60]
```

**第 4 轮循环：比较 50 和 60，50 较小，将节点 50 接到结果末尾。**

```text
连接后:

  resultHead → [10] → [20] → [30] → [40] → [50] → nullptr
  resultTail=[50]
  p=nullptr, q=[60]
```

**循环结束：p 为 nullptr，直接将 resultTail->next 指向 q（节点 60）。**

```text
  resultTail->next = q
  resultHead → [10] → [20] → [30] → [40] → [50] → [60] → nullptr
  // q 剩余部分已有序，整段直接接上
```

#### 12.5 带哨兵节点的实现

带哨兵节点（dummy node）可以省去"确定第一个节点"的特殊处理，代码更简洁。

In [ ]:
// 合并两个有序链表（带哨兵节点）

In [ ]:
Node* mergeSortedWithDummy(Node* headA, Node* headB) {
    Node dummy(0);         // 哨兵节点，值不重要
    Node* tail = &dummy;   // tail 始终指向结果链表的尾部
    Node* p = headA;
    Node* q = headB;

    while (p != nullptr && q != nullptr) {
        if (p->data <= q->data) {
            tail->next = p;
            tail = p;
            p = p->next;
        } else {
            tail->next = q;
            tail = q;
            q = q->next;
        }
    }

    tail->next = (p != nullptr) ? p : q;
    return dummy.next;  // 跳过哨兵节点
}

两个版本对比：

| 版本 | 首节点处理 | 尾指针 | 返回值 | 关键点 |
| ---- | ---------- | ------ | ------ | ------ |
| 不带哨兵 | 单独初始化 | `resultTail` | `resultHead` | 首节点需特殊处理 |
| 带哨兵 | 统一追加 | `tail` | `dummy.next` | 流程更统一简洁 |

哨兵节点在栈上分配，不属于结果链表，返回时取 `dummy.next` 即可。

```text
带哨兵节点流程示例:
  初始:  dummy → nullptr,  tail=&dummy
  追加 [10] 后:  dummy → [10] → nullptr,  tail=[10]
  返回:  dummy.next → [10] → [20] → ... → [60] → nullptr
```

#### 12.6 与 Concatenation 的区别

| 对比项 | Concatenation（连接）| Merge（合并）|
| ------ | -------------------- | ------------ |
| 前提 | 无序也可 | 两链均有序 |
| 动作 | A 尾接 B，1 次指针修改 | 每轮比较，逐节点选取 |
| 结果 | A 整段 + B 整段 | 按值交错的有序链 |
| 典型用途 | 直接拼接 | 保持有序 |

```text
同一组输入:

  A: [10] → [30] → [50] → nullptr
  B: [20] → [40] → [60] → nullptr

Concatenation: 只接尾巴，不看 data

  tailA=[50], headB=[20]
  结果: [10] → [30] → [50] → [20] → [40] → [60] → nullptr

Merge: 每轮比较当前节点，按值重排 next

  p=[10], q=[20], tail 每次接住较小节点
  结果: [10] → [20] → [30] → [40] → [50] → [60] → nullptr
```

| 轮次 | A 当前 | B 当前 | 选择 | 结果链 |
| ---- | ------ | ------ | ---- | ------ |
| 初始 | 10 | 20 | - | empty |
| 1 | 10 | 20 | A:10 | 10 |
| 2 | 30 | 20 | B:20 | 10 -> 20 |
| 3 | 30 | 40 | A:30 | 10 -> 20 -> 30 |
| 4 | 50 | 40 | B:40 | 10 -> 20 -> 30 -> 40 |
| 5 | 50 | 60 | A:50 | 10 -> 20 -> 30 -> 40 -> 50 |
| 收尾 | NULL | 60 | 剩余 B | 10 -> 20 -> 30 -> 40 -> 50 -> 60 |

#### 12.7 复杂度分析

| 操作 | 时间复杂度 | 说明 |
| ---- | ---------- | ---- |
| 比较并选取 | O(m+n) | 每个节点最多选一次 |
| 总计 | O(m+n) | m、n 是两条链长度 |

空间复杂度：O(1)。没有创建新节点，只是重新链接原有节点。

所有权提示：`mergeSorted` 返回的 `merged` 接管两条输入链的节点。原来的 `headA`、`headB` 只应视为别名，不能再作为两条独立链表分别释放。

---

### 13. 检测链表中的循环（Cycle Detection）

#### 13.1 什么是链表中的循环

正常链表的尾节点 `next` 指向 `NULL`。但某些情况下，某个节点的 `next` 可能指向链表中前面的节点，形成循环。

**正常的链表（无环）：**

```text
  head → [10] → [20] → [30] → [40] → nullptr
  // 沿 next 走，最终到达 nullptr
```

**有环的链表：**

```text
主链:
  head → [10] → [20] → [30] → [40]

回边:
  [40].next → [20]   // entry=[20]

遍历:
  10 → 20 → 30 → 40 → 20 → ...
```

环通常由编程错误（修改指针时不慎让某节点指向前面）或特殊数据结构设计（如约瑟夫环）造成。

**如何创建测试用的有环链表：**

In [ ]:
// 创建: 10 -> 20 -> 30 -> 40 -> 50 -> (回到 20)

In [ ]:
Node* createCyclicList() {
    Node* n1 = new Node(10);
    Node* n2 = new Node(20);
    Node* n3 = new Node(30);
    Node* n4 = new Node(40);
    Node* n5 = new Node(50);
    n1->next = n2;  n2->next = n3;  n3->next = n4;
    n4->next = n5;  n5->next = n2;  // 形成环
    return n1;
}

```text
  head → [10] → [20] → [30] → [40] → [50]
  [50].next → [20]  // entry=[20]，环长=4
```

#### 13.2 Floyd 循环检测算法（快慢指针）

**核心思想：** 慢指针（slow）每次走 1 步，快指针（fast）每次走 2 步。有环时 fast 必追上 slow；无环时 fast 先到达 nullptr。

**逐步图解（有环）：** 链表 10 → 20 → 30 → 40 → 50 → (回到 20)

```text
  head → [10] → [20] → [30] → [40] → [50]
  [50].next → [20]  // entry=[20]
  第 4 轮: slow=[50], fast=[50] → 相遇，有环
```

| 轮次 | `slow` | `fast` | 判断 |
| ---- | ------ | ------ | ---- |
| 初始 | 10 | 10 | 起点相同，先移动 |
| 1 | 20 | 30 | 未相遇 |
| 2 | 30 | 50 | 未相遇 |
| 3 | 40 | 30 | 未相遇 |
| 4 | 50 | 50 | 相遇，有环 |

```text
  第 4 轮: slow=[50], fast=[50]  // 相遇，有环
  entry=[20], [50].next → [20]
```

**无环情况：** 链表 10 -> 20 -> 30 -> NULL

```text
  初始: slow=[10], fast=[10]
  第 1 轮: slow=[20], fast=[30]
  第 2 轮: fast->next == nullptr，退出 // 无环
```

**为什么快指针一定能追上慢指针？** 进入环后，fast 相对 slow 每轮多走 1 步。从模环长的角度看，相对位置每轮变化 1，最多经过环长次一定重合。

**完整代码：**

In [ ]:
bool hasCycle(Node* head) {
    if (head == nullptr || head->next == nullptr) return false;

    Node* slow = head;
    Node* fast = head;

    while (fast != nullptr && fast->next != nullptr) {
        slow = slow->next;          // 慢指针走 1 步
        fast = fast->next->next;    // 快指针走 2 步
        if (slow == fast) return true;  // 相遇，有环
    }

    return false;  // fast 到达 NULL，无环
}

> 为什么条件是 `fast != nullptr && fast->next != nullptr`？因为 fast 每次走两步，必须保证 `fast` 和 `fast->next` 都非空，否则访问 `fast->next->next` 会段错误。

#### 13.3 找到环的入口节点

**数学推导：**

```text
  设 a = head→entry, b = entry→meet, c = 环长
  slow = a+b,  fast = a+b+n*c,  fast = 2*slow
  ⇒  a = (n-1)*c + (c-b)
  结论: head 走 a 步到 entry，meet 走 a 步也到 entry
        → 一指针放回 head，一指针留在 meet，同步走 1 步，首次相遇即入口
```

**两个验证示例：**

```text
  定义: a = head 到 entry； b = entry 到 meet； c = 环长
  相遇时: slow = a+b, fast = a+b+n*c
  因为 fast = 2*slow: 2(a+b) = a+b+n*c  ⇒  a = (n-1)*c + (c-b)
  几何含义:
    head 走 a 步 → entry
    meet 走 a 步也 → entry（多出的整圈回到同一点）
  结论:
    一个指针放回 head，一个留在 meet，同步走 1 步，第一次相遇即环入口。
```

```text
例1: 入口=[20], a=1, c=4, meet=[50]
  entry: [10]→[20]   slow: [50]→[20]   // 同步1步，均到 [20] ✓

例2: 入口=[30], a=2, c=3, meet=[40]
  entry: [10]→[20]→[30]   slow: [40]→[50]→[30]  // 同步2步，均到 [30] ✓
```

**完整代码：**

In [ ]:
Node* detectCycleEntry(Node* head) {
    if (head == nullptr || head->next == nullptr) return nullptr;

    Node* slow = head;
    Node* fast = head;

    while (fast != nullptr && fast->next != nullptr) {
        slow = slow->next;
        fast = fast->next->next;

        if (slow == fast) {
            // 一个指针从 head 出发，一个从相遇点出发
            Node* entry = head;
            while (entry != slow) {
                entry = entry->next;
                slow = slow->next;
            }
            return entry;  // 两者在入口相遇
        }
    }

    return nullptr;  // 无环
}

#### 13.4 计算环的长度

找到环中任意节点后，沿 `next` 走一圈回到自己，计数即得环长。
注意：参数必须是已经确认位于环内的节点，通常使用 `detectCycleEntry` 或 Floyd 相遇点作为输入。不要把普通无环链表节点直接传进来；普通 `deleteList` 也不能直接处理有环链表，否则会无限循环。

In [ ]:
int cycleLengthFromCycleNode(Node* node) {
    if (node == nullptr) return 0;
    Node* current = node->next;
    int length = 1;
    while (current != node) {
        current = current->next;
        length++;
    }
    return length;
}

**图解：** 假设相遇点在节点 40，环 40 -> 50 -> 20 -> 30 -> 40。

```text
  node=[40]
  环: [40] → [50] → [20] → [30] → 回 [40]
  初始: current=[50], length=1
```

| 轮次 | `current` | `length` | 判断 |
| ---- | --------- | -------- | ---- |
| 1 | 50 | 1 | 未回到起点 |
| 2 | 20 | 2 | 未回到起点 |
| 3 | 30 | 3 | 未回到起点 |
| 4 | 40 | 4 | 回到起点，停止 → 环长 = 4 |

**完整检测环的程序：**

In [ ]:
#include <iostream>
using namespace std;

In [ ]:
struct Node {
    int data;
    Node* next;
    Node(int val) : data(val), next(nullptr) {}
};

In [ ]:
bool hasCycle(Node* head) {
    if (head == nullptr || head->next == nullptr) return false;
    Node* slow = head, *fast = head;
    while (fast != nullptr && fast->next != nullptr) {
        slow = slow->next;
        fast = fast->next->next;
        if (slow == fast) return true;
    }
    return false;
}

In [ ]:
Node* detectCycleEntry(Node* head) {
    if (head == nullptr || head->next == nullptr) return nullptr;
    Node* slow = head, *fast = head;
    while (fast != nullptr && fast->next != nullptr) {
        slow = slow->next;
        fast = fast->next->next;
        if (slow == fast) {
            Node* entry = head;
            while (entry != slow) {
                entry = entry->next;
                slow = slow->next;
            }
            return entry;
        }
    }
    return nullptr;
}

In [ ]:
int cycleLengthFromCycleNode(Node* node) {
    if (node == nullptr) return 0;
    Node* current = node->next;
    int length = 1;
    while (current != node) {
        current = current->next;
        length++;
    }
    return length;
}

In [ ]:
void deleteList(Node* head) {
    while (head != nullptr) {
        Node* nextNode = head->next;
        delete head;
        head = nextNode;
    }
}

In [ ]:
int main() {
    Node* n1 = new Node(10);
    Node* n2 = new Node(20);
    Node* n3 = new Node(30);
    Node* n4 = new Node(40);
    Node* n5 = new Node(50);
    n1->next = n2;  n2->next = n3;  n3->next = n4;
    n4->next = n5;  n5->next = n2;  // 形成环

    cout << "是否有环: " << (hasCycle(n1) ? "是" : "否") << endl;
    Node* entry = detectCycleEntry(n1);
    if (entry) cout << "环入口值: " << entry->data << endl;
    cout << "环长度: " << cycleLengthFromCycleNode(entry) << endl;

    n5->next = nullptr;  // 释放前先断环，否则普通 deleteList 会无限循环
    deleteList(n1);
    return 0;
}

In [ ]:
main();

输出：

```text
是否有环: 是
环入口值: 20
环长度: 4
```

#### 13.5 复杂度分析

| 算法 | 时间 | 空间 | 核心动作 |
| ---- | ---- | ---- | -------- |
| 检测有环 | O(n) | O(1) | 快慢指针相遇 |
| 找环入口 | O(n) | O(1) | 两指针同步走 |
| 计算环长 | O(n) | O(1) | 沿环走一圈 |

三个操作结合，总时间复杂度 O(n)，空间复杂度 O(1)。

---

### 14. 链表的 C++ 类封装

#### 14.1 为什么要封装

此前我们用全局函数 + `head` 指针操作链表，存在以下问题：

**问题 1: 调用者需要自己管理 head 指针**

In [ ]:
Node* head = nullptr;
head = insertAtBeginning(head, 10);
head = insertAtEnd(head, 20);
head = deleteNode(head, 10);
// 容易忘记更新 head，特别是删除头节点时

**问题 2: 容易误操作**

In [ ]:
head = nullptr;
// 一不小心就把整个链表"丢了"
head->next = nullptr;
// 一不小心就截断了链表

**问题 3: 没有统一的清理机制**

```text
// 忘记调用 deleteList(head) → 内存泄漏
// 多次调用 → 节点被重复释放
```

**封装的好处：**

- `head` 是类的私有成员，外部无法直接访问或误修改
- 所有操作通过公共接口进行，安全且统一
- 析构函数自动清理内存，不会忘记释放
- 代码组织更清晰，一个类就是一个完整的数据结构

#### 14.2 完整的链表类实现

In [ ]:
#include <iostream>
#include <stdexcept>
using namespace std;

In [ ]:
class LinkedList {
private:
    struct Node {
        int data;
        Node* next;
        Node(int val) : data(val), next(nullptr) {}
    };
    Node* head;

public:
    LinkedList() : head(nullptr) {}

    ~LinkedList() {
        while (head) {
            Node* temp = head->next;
            delete head;
            head = temp;
        }
    }

    LinkedList(const LinkedList&) = delete;
    LinkedList& operator=(const LinkedList&) = delete;
    LinkedList(LinkedList&&) = delete;
    LinkedList& operator=(LinkedList&&) = delete;

    void display() const {
        for (Node* t = head; t; t = t->next) cout << t->data << " -> ";
        cout << "NULL\n";
    }

    void insertAtBeginning(int val) {
        Node* newNode = new Node(val);
        newNode->next = head;
        head = newNode;
    }

    void insertAtEnd(int val) {
        if (!head) { head = new Node(val); return; }
        Node* temp = head;
        while (temp->next) temp = temp->next;
        temp->next = new Node(val);
    }

    void insertAt(int pos, int val) {
        if (pos < 0) throw out_of_range("位置不能为负数");
        if (pos == 0) return insertAtBeginning(val);
        Node* temp = head;
        for (int i = 0; i < pos - 1 && temp; ++i) temp = temp->next;
        if (!temp) throw out_of_range("位置超出链表长度");
        Node* newNode = new Node(val);
        newNode->next = temp->next;
        temp->next = newNode;
    }

    bool deleteNode(int val) {
        if (!head) return false;
        if (head->data == val) {
            Node* temp = head; head = head->next; delete temp; return true;
        }
        Node* temp = head;
        while (temp->next && temp->next->data != val) temp = temp->next;
        if (!temp->next) return false;
        Node* toDelete = temp->next;
        temp->next = toDelete->next;
        delete toDelete;
        return true;
    }

    bool deleteAt(int pos) {
        if (!head || pos < 0) return false;
        if (pos == 0) {
            Node* temp = head; head = head->next; delete temp; return true;
        }
        Node* temp = head;
        for (int i = 0; i < pos - 1 && temp->next; ++i) temp = temp->next;
        if (!temp->next) return false;
        Node* toDelete = temp->next;
        temp->next = toDelete->next;
        delete toDelete;
        return true;
    }

    bool search(int val) const {
        for (Node* t = head; t; t = t->next)
            if (t->data == val) return true;
        return false;
    }

    int length() const {
        int count = 0;
        for (Node* t = head; t; t = t->next) count++;
        return count;
    }

    int sum() const {
        int total = 0;
        for (Node* t = head; t; t = t->next) total += t->data;
        return total;
    }

    int max() const {
        if (!head) throw runtime_error("链表为空");
        int maxVal = head->data;
        for (Node* t = head->next; t; t = t->next)
            if (t->data > maxVal) maxVal = t->data;
        return maxVal;
    }

    void reverse() {
        Node* prev = nullptr;
        while (head) {
            Node* next = head->next;
            head->next = prev;
            prev = head;
            head = next;
        }
        head = prev;
    }

    bool isSorted() const {
        if (!head) return true;
        for (Node* t = head; t->next; t = t->next)
            if (t->data > t->next->data) return false;
        return true;
    }

    // 只删除相邻重复节点，调用前应保证链表已排序
    void removeDuplicates() {
        Node* curr = head;
        while (curr && curr->next) {
            if (curr->data == curr->next->data) {
                Node* temp = curr->next;
                curr->next = temp->next;
                delete temp;
            } else curr = curr->next;
        }
    }

    void concatenate(LinkedList& other) {
        if (this == &other) return;  // 防止自连接形成环或丢失所有权
        if (!other.head) return;
        if (!head) { head = other.head; other.head = nullptr; return; }
        Node* tail = head;
        while (tail->next) tail = tail->next;
        tail->next = other.head;
        other.head = nullptr;
    }

    void mergeSorted(LinkedList& other) {
        if (this == &other) return;  // 自合并没有意义，且会破坏链表结构
        Node dummy(0);
        Node* tail = &dummy;
        Node* p = head, *q = other.head;
        while (p && q) {
            if (p->data <= q->data) { tail->next = p; p = p->next; }
            else { tail->next = q; q = q->next; }
            tail = tail->next;
        }
        tail->next = p ? p : q;
        head = dummy.next;
        other.head = nullptr;
    }
};

**类的设计要点说明：**

| 设计点 | 做法 | 目的 |
| ------ | ---- | ---- |
| 隐藏节点 | `Node` 放在 `private` | 外部不能乱改 `next` |
| 保护入口 | `head` 私有 | 只能由成员函数维护 |
| 明确空表 | 构造时 `head=nullptr` | 初始状态清楚 |
| 自动释放 | 析构函数逐节点 `delete` | 避免内存泄漏 |
| 禁用拷贝 | 删除拷贝构造/赋值 | 避免重复释放 |
| 转移所有权 | `other.head=nullptr` | 只让一个对象负责释放 |
| 只读接口 | 使用 `const` 成员函数 | 不修改链表结构 |

```text
封装后的对象边界:
  外部代码 → public 方法
  LinkedList list 内部: private head → [10] → [20] → [30] → nullptr
```

```text
析构函数的 ownership 关系:
  list 拥有: list.head → [10] → [20] → [30] → nullptr
  ~LinkedList() 删除: 逐个 delete [10], [20], [30]
  注意: 若两对象的 head 指向同一条链，析构时会重复 delete。
```

#### 14.3 使用示例

In [ ]:
int main() {
    LinkedList list;
    for (int v : {10, 20, 30}) list.insertAtEnd(v);
    list.insertAtBeginning(5);
    list.insertAt(2, 15);
    list.display();       // 5 -> 10 -> 15 -> 20 -> 30 -> NULL
    cout << "长度=" << list.length() << " 总和=" << list.sum() << " 最大=" << list.max() << endl;

    cout << "查找 15: " << (list.search(15) ? "找到" : "未找到") << endl;
    list.deleteNode(15);
    list.deleteAt(0);
    list.display();       // 10 -> 20 -> 30 -> NULL

    list.reverse();
    list.display();       // 30 -> 20 -> 10 -> NULL

    LinkedList sorted;
    for (int v : {1, 2, 2, 3, 3, 3}) sorted.insertAtEnd(v);
    sorted.display();     // 1 -> 2 -> 2 -> 3 -> 3 -> 3 -> NULL
    sorted.removeDuplicates();
    sorted.display();     // 1 -> 2 -> 3 -> NULL

    LinkedList a, b;
    for (int v : {1, 2, 3}) a.insertAtEnd(v);
    for (int v : {4, 5})    b.insertAtEnd(v);
    a.concatenate(b);
    a.display();          // 1 -> 2 -> 3 -> 4 -> 5 -> NULL

    LinkedList m, n;
    for (int v : {1, 3, 5}) m.insertAtEnd(v);
    for (int v : {2, 4, 6}) n.insertAtEnd(v);
    m.mergeSorted(n);
    m.display();          // 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> NULL

    return 0;
}

In [ ]:
main();

**封装前后对比：**

```text
封装前（全局函数）:
  head → [10] → [20] → nullptr
  风险: head 可被外部直接改坏或忘记 delete

封装后（类）:
  list.private head → [10] → [20] → nullptr
  外部只能调用方法，析构时自动释放。
```

```text
concatenate 后的所有权转移:
  连接前: a.head → [1] → [2],  b.head → [3] → [4]
  连接后: a.head → [1] → [2] → [3] → [4],  b.head → nullptr
  转移: b 变空，节点已归 a，析构 b 时不删除这些节点。
```

```text
mergeSorted 后的所有权转移:
  合并前: m.head → [1] → [3],  n.head → [2] → [4]
  合并后: m.head → [1] → [2] → [3] → [4],  n.head → nullptr
  转移: 不复制节点，n 的节点转移给 m，必须清空 n.head。
```

封装是 C++ 面向对象的核心思想。通过将数据（head）和操作绑定在一起，得到安全、易用、自管理的链表数据结构。后续的栈、队列、树等数据结构都会采用类似的封装方式。

## 第五阶段：循环链表（Circular Linked List）

循环链表是单链表的变体，尾节点不再指向 nullptr，而是指回头节点。

本阶段默认链表结构正确：非空时沿 `next` 一定能回到 `head`。若指针损坏，遍历可能无法结束，实际工程中应额外设置保护条件。

---

### 15. 循环链表的概念与创建

#### 15.1 什么是循环链表

循环链表与普通单链表的唯一区别：**尾节点的 `next` 指回头节点**，形成闭环。

**普通单链表的结构：**

```text
普通单链表:
  [1] → [2] → [3] → [4] → nullptr
  tail->next == nullptr，走到 tail 后结束。
```

尾节点（值为 4）的 `next` 是 `nullptr`，链表到此结束。

**循环链表的结构：**

```text
循环链表:
  [1] → [2] → [3] → [4] → head
  tail->next == head，走到 tail 后回到 head。
```

尾节点（值为 4）的 `next` 指回了头节点（值为 1），形成闭环。

**核心特点：**

- 从任何节点出发，沿 `next` 最终都能回到起点
- 非空循环链表内部没有 `nullptr`，遍历时必须自行判断何时停止
- 空循环链表：`head == nullptr`

**不变量检查表：**

| 情况 | 必须满足 |
| ---- | -------- |
| 空表 | `head == nullptr` |
| 单节点环 | `head->next == head` |
| 多节点环 | 存在尾节点 `tail`，且 `tail->next == head` |
| 遍历停止 | 再次回到起点，而不是遇到 `nullptr` |

**循环链表 vs 普通链表对比：**

```text
  普通: [1] → [2] → [3] → nullptr  (遇 nullptr 结束)
  循环: [1] → [2] → [3] → head     (经 next 回起点)
```

**什么时候用循环链表？**

- 约瑟夫环问题（N 个人围成一圈报数）
- 操作系统中的轮转调度（Round-Robin Scheduling）
- 需要反复循环处理数据的场景（如音乐播放列表循环播放）

#### 15.2 循环链表的创建（尾插法）

循环链表的创建和普通链表基本相同，唯一区别：**创建完成后，把尾节点的 `next` 指回 `head`**。

**思路分析：**

1. 逐个读入数据，用尾插法创建节点并链接
2. 全部插入完成后，令 `tail->next = head`，形成环

**完整代码：**

In [ ]:
#include <iostream>
using namespace std;

// 节点结构体（与单链表完全相同）

In [ ]:
struct Node {
    int data;
    Node* next;
    Node(int val) : data(val), next(nullptr) {}
};

In [ ]:
// 用尾插法创建循环链表
Node* createCircularList(int arr[], int n) {
    if (n <= 0) return nullptr;

    Node* head = new Node(arr[0]);
    Node* tail = head;

    for (int i = 1; i < n; i++) {
        tail->next = new Node(arr[i]);
        tail = tail->next;
    }

    // 关键步骤：尾节点 next 指回头节点，形成环
    tail->next = head;

    return head;
}

In [ ]:
void deleteCircularList(Node* head) {
    if (head == nullptr) return;

    Node* curr = head->next;
    while (curr != head) {
        Node* nextNode = curr->next;
        delete curr;
        curr = nextNode;
    }
    delete head;
}

In [ ]:
int main() {
    int arr[] = {1, 2, 3, 4, 5};
    int n = 5;
    Node* head = createCircularList(arr, n);

    // 此时 head 指向值为 1 的节点，尾节点（值为 5）的 next 指回 head
    if (head == nullptr) return 0;
    cout << "head->data = " << head->data << endl;
    // 验证循环：尾节点的 next 应该等于 head
    Node* tail = head;
    while (tail->next != head) {
        tail = tail->next;
    }
    cout << "tail->data = " << tail->data << endl;
    cout << "tail->next == head ? " << (tail->next == head ? "true" : "false") << endl;

    deleteCircularList(head);
    return 0;
}

In [ ]:
main();

```text
输出：
head->data = 1
tail->data = 5
tail->next == head ? true
```

**逐步图解（创建过程）：**

第 1 步：创建第一个节点

```text
创建第一个节点:
  head=[1], tail=[1],  [1] → nullptr
```

第 2 步：插入值为 2 的节点（普通尾插）

```text
插入节点 2 (普通尾插):
  head=[1], tail=[2],  [1] → [2] → nullptr
```

第 3 步：插入值为 3 的节点（普通尾插）

```text
插入节点 3 (普通尾插):
  head=[1], tail=[3],  [1] → [2] → [3] → nullptr
```

第 4 步：`tail->next = head`，形成环

```text
执行 tail->next = head 形成环:
  head=[1], tail=[3],  [1] → [2] → [3] → head
```

尾节点的 `next` 从 `nullptr` 变成了 `head`，环形成。

**复杂度分析：**

| 操作 | 时间复杂度 | 说明 |
| ---- | ---------- | ---- |
| 创建 n 个节点 | O(n) | 遍历数组一次 |
| 尾节点指回头节点 | O(1) | `tail->next = head` |
| 总计 | O(n) | 主要成本是建节点 |

---

### 16. 显示循环链表

#### 16.1 迭代显示

**核心问题：** 普通链表用 `while (curr != nullptr)` 遍历，但循环链表没有 `nullptr`，这样写会**死循环**。

**错误写法（会死循环！）：**

In [ ]:
// 错误！循环链表中没有 nullptr，这个循环永远停不下来

In [ ]:
void displayWrong(Node* head) {
    Node* curr = head;
    while (curr != nullptr) {  // 永远为 true，死循环！
        cout << curr->data << " ";
        curr = curr->next;
    }
}

**正确做法：** 使用 `do-while` 循环，先处理当前节点，再前进，当 `curr` 回到 `head` 时停止。

**为什么用 `do-while`？** 若用 `while (curr != head)` 且链表非空，`curr` 初始就是 `head`，条件一开始就为 `false`，循环一次都不执行。`do-while` 先执行后判断，保证至少访问一次，走完一圈回到 `head` 时停止。

**完整代码：**

In [ ]:
#include <iostream>
using namespace std;

In [ ]:
struct Node {
    int data;
    Node* next;
    Node(int val) : data(val), next(nullptr) {}
};

In [ ]:
// 迭代显示循环链表
void displayCircular(Node* head) {
    if (head == nullptr) {
        cout << "链表为空" << endl;
        return;
    }

    Node* curr = head;
    do {
        cout << curr->data << " -> ";
        curr = curr->next;
    } while (curr != head);  // 回到 head 时停止

    cout << "(回到头节点 " << head->data << ")" << endl;
}

In [ ]:
void deleteCircularList(Node* head) {
    if (head == nullptr) return;
    Node* curr = head->next;
    while (curr != head) {
        Node* nextNode = curr->next;
        delete curr;
        curr = nextNode;
    }
    delete head;
}

In [ ]:
int main() {
    int arr[] = {10, 20, 30, 40};
    int n = 4;

    // 手动创建循环链表
    Node* head = new Node(arr[0]);
    Node* tail = head;
    for (int i = 1; i < n; i++) {
        tail->next = new Node(arr[i]);
        tail = tail->next;
    }
    tail->next = head;  // 形成环

    displayCircular(head);
    deleteCircularList(head);
    return 0;
}

In [ ]:
main();

```text
输出：
10 -> 20 -> 30 -> 40 -> (回到头节点 10)
```

**逐步图解（遍历过程）：**

初始状态，`curr = head`：

```text
初始状态:
  head=[10], tail=[40], curr=[10]
  [10] → [20] → [30] → [40] → head
  do...while 循环会先输出 curr，即 head 节点。
```

第 1 次循环：输出 `10`，`curr` 前进到 20

```text
第 1 次循环后:
  curr=[20], 条件 curr != head 成立，继续遍历。
```

第 2 次循环：输出 `20`，`curr` 前进到 30
第 3 次循环：输出 `30`，`curr` 前进到 40
第 4 次循环：输出 `40`，`curr` 前进到 10（即 `head`）

```text
第 4 次循环后:
  curr 经 tail->next 回到 head，条件 curr != head 为假，循环停止。
```

此时 `curr == head`，条件 `curr != head` 为 `false`，循环结束。

**复杂度：** O(n)，每个节点访问一次。

#### 16.2 递归显示

递归显示循环链表需要额外的参数来记录 `head`。普通链表用 `curr == nullptr` 终止；循环链表通常在“当前节点是尾节点（`curr->next == head`）”时停止。

**完整代码：**

In [ ]:
// 递归显示循环链表
// curr：当前节点，head：头节点（用于判断是否走完一圈）

In [ ]:
void displayRecursive(Node* curr, Node* head) {
    // 防御性检查：空链表直接返回
    if (curr == nullptr) return;

    // 终止条件：当前节点是尾节点，输出后停止
    if (curr != head && curr->next == head) {
        // 当前是最后一个节点，输出后递归回来打印 head 标记
        cout << curr->data << " -> (回到头节点 " << head->data << ")" << endl;
        return;
    }

    // 单节点环：head 自己就是尾节点
    if (curr == head && curr->next == head) {
        // 只有一个节点的情况
        cout << curr->data << " -> (回到头节点 " << head->data << ")" << endl;
        return;
    }

    cout << curr->data << " -> ";
    displayRecursive(curr->next, head);
}

// 更简洁的写法：不需要单独的 visited / first-call 标记

In [ ]:
void displayRecursiveSimple(Node* curr, Node* head) {
    if (curr == nullptr) return;

    cout << curr->data << " -> ";

    // 只要下一个不是 head，就继续递归
    if (curr->next != head) {
        displayRecursiveSimple(curr->next, head);
    } else {
        cout << "(回到头节点 " << head->data << ")" << endl;
    }
}

**递归过程图解：**

```text
递归遍历:
  [10] → [20] → [30] → head
  curr 每层后移，head 固定用于判断是否完成一圈。
```

```text
递归深入过程:
  display(curr=[10], head=[10]) → 输出 10
    display(curr=[20], head=[10]) → 输出 20
      display(curr=[30], head=[10]) → 输出 30
  当 curr->next == head 时到达尾节点。
```

```text
停止条件触发:
  curr=[30], curr->next == head。
  输出 30 并打印“回到头节点”，递归结束。
```

**复杂度：** O(n)，递归深度为 n，需要 O(n) 的栈空间。循环链表显示优先使用迭代版；递归版在大链表上可能栈溢出。

---

### 17. 在循环链表中插入

#### 17.1 在头部插入

在循环链表头部插入比普通链表多一步：**需找到尾节点，让其 `next` 指向新节点**。

**思路分析：**

1. 创建新节点 `newNode`
2. 新节点的 `next` 指向原来的 `head`
3. 找到尾节点（`tail->next == head` 的那个节点）
4. 尾节点的 `next` 指向新节点
5. 更新 `head = newNode`

**特殊情况：** 空链表时，新节点自己指向自己。

**完整代码：**

In [ ]:
// 在循环链表头部插入

In [ ]:
Node* insertAtHead(Node* head, int val) {
    Node* newNode = new Node(val);

    // 情况 1：空链表
    if (head == nullptr) {
        newNode->next = newNode;  // 自己指向自己，形成单节点环
        return newNode;
    }

    // 情况 2：非空链表
    // 第一步：找到尾节点
    Node* tail = head;
    while (tail->next != head) {
        tail = tail->next;
    }

    // 第二步：新节点 next 指向原 head
    newNode->next = head;

    // 第三步：尾节点 next 指向新节点
    tail->next = newNode;

    // 第四步：更新 head
    return newNode;
}

**逐步图解（非空链表）：**

初始状态（循环链表：10 -> 20 -> 30）：

```text
初始状态:
  head=[10], tail=[30],  [10] → [20] → [30] → head
```

第 1 步：创建新节点（值为 5）

```text
创建新节点:
  newNode=[5], 此时尚未接入链表。
```

第 2 步：`newNode->next = head`（新节点指向原头节点 10）

```text
新节点指向原 head:
  newNode->next = head;
  [5] → [10] → [20] → [30] → head
  此时 head 仍指向原头节点 [10]，tail 仍指向尾节点 [30]。
```

第 3 步：`tail->next = newNode`（尾节点 30 指向新节点 5）

```text
尾节点指向新节点:
  tail->next = newNode;
  [5] → [10] → [20] → [30] → newNode
  环已闭合，但 head 指针未更新。
```

第 4 步：`head = newNode`（更新头指针）

```text
更新 head:
  head = newNode;
  最终链表: [5] → [10] → [20] → [30] → head
```

完成！新链表为：5 -> 10 -> 20 -> 30（循环）。

**复杂度：** O(n)，因为需要遍历找到尾节点。

> 注意：如果维护一个 `tail` 指针（或用双向循环链表），可以优化到 O(1)。

#### 17.2 在指定位置插入

**思路分析：**

在位置 `pos`（从 0 开始）插入新节点：
1. 如果 `pos == 0`，相当于在头部插入
2. 否则，找到第 `pos - 1` 个节点（前驱节点），在它后面插入

**完整代码：**

In [ ]:
// 在循环链表的指定位置插入

In [ ]:
Node* insertAtPosition(Node* head, int val, int pos) {
    if (pos < 0) {
        cout << "位置不能为负数" << endl;
        return head;
    }

    Node* newNode = new Node(val);

    // 空链表：只有 pos == 0 才能插入
    if (head == nullptr) {
        if (pos == 0) {
            newNode->next = newNode;
            return newNode;
        }
        cout << "位置 " << pos << " 超出范围" << endl;
        delete newNode;
        return nullptr;
    }

    // 在头部插入
    if (pos == 0) {
        // 找尾节点
        Node* tail = head;
        while (tail->next != head) {
            tail = tail->next;
        }
        newNode->next = head;
        tail->next = newNode;
        return newNode;
    }

    // 找到第 pos-1 个节点
    Node* curr = head;
    for (int i = 0; i < pos - 1; i++) {
        curr = curr->next;
        // 如果已经回到 head，说明 pos 超出范围
        if (curr == head) {
            cout << "位置 " << pos << " 超出范围" << endl;
            delete newNode;
            return head;
        }
    }

    // 在 curr 后面插入 newNode
    newNode->next = curr->next;
    curr->next = newNode;

    return head;
}

**图解（在中间位置插入）：**

假设链表为 10 -> 20 -> 30 -> 40（循环），在 `pos = 2` 插入 25。

```text
找到前驱节点 prev:
  prev(代码中的 curr)=[20], 新节点要插到 prev 后面。
  [10] → [20] → [30] → [40] → head
```

```text
新节点接住后继:
  newNode->next = curr->next;
  [25] → [30]
```

```text
前驱接上新节点:
  curr->next = newNode;
  最终链表: [10] → [20] → [25] → [30] → [40] → head
```

**复杂度：** O(n)，最坏情况需要遍历到尾节点。

#### 17.3 在有序循环链表中插入

给定一个升序排列的循环链表，插入新节点后仍保持升序。

**三种情况：**

1. **空链表：** 新节点自己成环
2. **新节点比所有节点都小（或等于最小）：** 插入到头部之前
3. **新节点插在中间或尾部：** 找到第一个比它大的节点，插在前面

**完整代码：**

In [ ]:
// 在有序循环链表中插入（保持升序）

In [ ]:
Node* insertSorted(Node* head, int val) {
    Node* newNode = new Node(val);

    // 情况 1：空链表
    if (head == nullptr) {
        newNode->next = newNode;
        return newNode;
    }

    // 情况 2：val <= head->data，新节点成为新的头节点
    if (val <= head->data) {
        // 找尾节点
        Node* tail = head;
        while (tail->next != head) {
            tail = tail->next;
        }
        newNode->next = head;
        tail->next = newNode;
        return newNode;  // 新节点成为新 head
    }

    // 情况 3：在链表中间或尾部找到合适位置
    Node* curr = head;
    // 找到第一个比 val 大的节点的前驱
    while (curr->next != head && curr->next->data <= val) {
        curr = curr->next;
    }

    // 在 curr 后面插入
    newNode->next = curr->next;
    curr->next = newNode;

    return head;
}

**图解（情况 3，在中间插入）：**

有序循环链表：10 -> 20 -> 30 -> 40（循环），插入 25。

第 1 步：从 head 开始，找第一个 `data > 25` 的节点的前驱。

```text
查找插入位置:
  curr=[10], curr->next->data(20) <= 25，curr 后移。
```

```text
继续查找:
  curr=[20], curr->next->data(30) > 25，停止。
  curr 即为插入位置的前驱。
```

第 2 步：在 20 后面插入 25。

```text
执行插入:
  newNode->next = curr->next;
  curr->next = newNode;
```

```text
插入完成:
  [10] → [20] → [25] → [30] → [40] → head
```

**复杂度：** O(n)，最坏情况插入到尾部。

---

### 18. 从循环链表中删除

#### 18.1 删除头节点

删除头节点比普通链表多一步：**需要找到尾节点，让尾节点的 `next` 指向新的头节点**。

**思路分析：**

1. 如果链表为空，无法删除
2. 如果链表只有一个节点，删除后 `head = nullptr`
3. 否则：找到尾节点，先保存 `newHead = head->next`，再令 `tail->next = newHead`，最后释放原 `head`

**完整代码：**

In [ ]:
// 删除循环链表的头节点

In [ ]:
Node* deleteHead(Node* head) {
    // 空链表
    if (head == nullptr) {
        cout << "链表为空，无法删除" << endl;
        return nullptr;
    }

    // 只有一个节点
    if (head->next == head) {
        delete head;
        return nullptr;
    }

    // 多个节点
    // 第一步：找到尾节点
    Node* tail = head;
    while (tail->next != head) {
        tail = tail->next;
    }

    // 第二步：尾节点 next 指向新头节点
    Node* newHead = head->next;
    tail->next = newHead;

    // 第三步：释放原头节点
    delete head;

    // 第四步：返回新头节点
    return newHead;
}

**图解（多节点情况）：**

删除前（链表：10 -> 20 -> 30，循环）：

```text
初始状态:
  head=[10], tail=[30]
  [10] → [20] → [30] → head
```

第 1 步：找到尾节点（值为 30）

```text
找到尾节点:
  tail=[30], tail->next == head
```

第 2 步：`tail->next = head->next`（尾节点指向 20）

```text
尾节点指向新 head:
  newHead = head->next;
  tail->next = newHead;
  此时原 head [10] 已从链表摘除。
```

第 3 步：释放原头节点（值为 10）

```text
释放节点并返回:
  delete 原 head，返回 newHead。
  最终链表: [20] → [30] → head
```

完成！新链表为：20 -> 30（循环）。

**复杂度：** O(n)，需要遍历找到尾节点。

#### 18.2 删除指定位置的节点

**思路分析：**

删除位置 `pos`（从 0 开始）的节点：
1. 如果 `pos == 0`，相当于删除头节点
2. 否则，找到第 `pos - 1` 个节点（前驱），删除它的下一个节点

**完整代码：**

In [ ]:
// 删除循环链表指定位置的节点

In [ ]:
Node* deleteAtPosition(Node* head, int pos) {
    if (pos < 0) {
        cout << "位置不能为负数" << endl;
        return head;
    }

    if (head == nullptr) {
        cout << "链表为空" << endl;
        return nullptr;
    }

    // 删除头节点
    if (pos == 0) {
        return deleteHead(head);
    }

    // 找到第 pos-1 个节点
    Node* curr = head;
    for (int i = 0; i < pos - 1; i++) {
        curr = curr->next;
        if (curr == head) {
            cout << "位置 " << pos << " 超出范围" << endl;
            return head;
        }
    }

    // 检查要删除的节点是否存在
    if (curr->next == head) {
        cout << "位置 " << pos << " 超出范围" << endl;
        return head;
    }

    // 删除 curr->next
    Node* toDelete = curr->next;
    curr->next = toDelete->next;
    delete toDelete;

    return head;
}

**图解（删除中间位置）：**

假设链表为 10 -> 20 -> 30 -> 40（循环），删除 `pos = 2` 的节点 [30]。

```text
找到前驱节点和待删除节点:
  curr=[20], toDelete=[30]
```

```text
执行删除:
  curr->next = toDelete->next;
  delete toDelete;
  最终链表: [10] → [20] → [40] → head
```

**复杂度：** O(n)，最坏情况遍历到倒数第二个节点。

#### 18.3 按值删除节点

删除循环链表中第一个值等于 `val` 的节点。

**思路分析：**

1. 如果 `head->data == val`，删除头节点
2. 否则遍历链表，找到值为 `val` 的节点的前驱，然后删除

**完整代码：**

In [ ]:
// 按值删除循环链表中的节点

In [ ]:
Node* deleteByValue(Node* head, int val) {
    if (head == nullptr) {
        cout << "链表为空" << endl;
        return nullptr;
    }

    // 要删除的是头节点
    if (head->data == val) {
        return deleteHead(head);
    }

    // 遍历找值为 val 的节点的前驱
    Node* curr = head;
    while (curr->next != head) {
        if (curr->next->data == val) {
            Node* toDelete = curr->next;
            curr->next = toDelete->next;
            delete toDelete;
            cout << "已删除值为 " << val << " 的节点" << endl;
            return head;
        }
        curr = curr->next;
    }

    cout << "未找到值为 " << val << " 的节点" << endl;
    return head;
}

**图解（删除值为 20 的节点）：**

删除前：

```text
删除前:

head=[10], tail=[40]
[10] → [20] → [30] → [40] → head
prev(代码中的 curr)=[10], toDelete=[20]

含义:
  curr->next->data == 20；
  toDelete = curr->next。
```

`curr = 10`，`curr->next->data == 20 == val`，找到目标。

```text
删除后:

head=[10], tail=[40], prev(代码中的 curr)=[10]
[10] → [30] → [40] → head
toDelete=[20]  (已摘出，随后 delete)

含义:
  curr->next = toDelete->next；
  本例删除的是中间节点，head 和 tail 不变。
```

删除后：

```text
删除完成:
  [10] → [30] → [40] → head
  节点 [20] 已释放。
```

完成！链表变为：10 -> 30 -> 40（循环）。

**复杂度：** O(n)，最坏情况遍历整条链表。

---

## 第六阶段：双向链表（Doubly Linked List）

### 19. 双向链表的结构与创建

#### 19.1 为什么需要双向链表

单链表有一个明显缺点：**只能向后走，不能向前走**。若当前指向某节点，想找前驱只能从 head 重新遍历。双向链表通过增加 `prev` 指针，可 O(1) 访问前驱。

**单链表的前驱访问问题：**

```text
单链表找前驱困难:
  head → [10] → [20] → [30] → [40] → nullptr
  若 current 在 [30]，想找前驱 [20] 只能从 head 重头遍历。
```

如果有 `prev` 指针，就能直接 O(1) 访问前驱。

**双向链表的优势：**

```text
双向链表 O(1) 访问前后驱:
  head → [10] ⇄ [20] ⇄ [30] ⇄ [40] ← tail
  [30]->prev == [20], [30]->next == [40]
```

#### 19.2 节点定义

双向链表的每个节点有三部分：`prev`（前驱指针）、`data`（数据）、`next`（后继指针）。

In [ ]:
struct DNode {
    int data;
    DNode* prev;  // 指向前一个节点
    DNode* next;  // 指向后一个节点
    DNode(int val) : data(val), prev(nullptr), next(nullptr) {}
};

**内存结构图：**

```text
内存结构:
  head → [10] ⇄ [20] ⇄ [30] ← tail
  [10].prev == nullptr, [30].next == nullptr
```

更清晰的示意：

```text
相邻一致性:
  若 a 后面是 b，则 a.next == b 且 b.prev == a
```

#### 19.3 双向链表的创建（尾插法）

**思路分析：**

1. 创建第一个节点，`head` 和 `tail` 都指向它
2. 每次插入新节点时：
   - `tail->next = newNode`（旧尾节点的 next 指向新节点）
   - `newNode->prev = tail`（新节点的 prev 指向旧尾节点）
   - `tail = newNode`（更新 tail）

**完整代码：**

In [ ]:
#include <iostream>
using namespace std;

In [ ]:
struct DNode {
    int data;
    DNode* prev;
    DNode* next;
    DNode(int val) : data(val), prev(nullptr), next(nullptr) {}
};

In [ ]:
// 尾插法创建双向链表
DNode* createDoublyList(int arr[], int n) {
    if (n <= 0) return nullptr;

    DNode* head = new DNode(arr[0]);
    DNode* tail = head;

    for (int i = 1; i < n; i++) {
        DNode* newNode = new DNode(arr[i]);
        tail->next = newNode;  // 旧尾节点 next 指向新节点
        newNode->prev = tail;  // 新节点 prev 指向旧尾节点
        tail = newNode;        // 更新 tail
    }

    return head;
}

In [ ]:
void deleteDoublyList(DNode* head) {
    while (head != nullptr) {
        DNode* nextNode = head->next;
        delete head;
        head = nextNode;
    }
}

// 显示双向链表

In [ ]:
void display(DNode* head) {
    DNode* curr = head;
    while (curr != nullptr) {
        cout << curr->data << " <-> ";
        curr = curr->next;
    }
    cout << "NULL" << endl;
}

In [ ]:
int main() {
    int arr[] = {10, 20, 30, 40, 50};
    int n = 5;
    DNode* head = createDoublyList(arr, n);
    display(head);
    deleteDoublyList(head);
    return 0;
}

In [ ]:
main();

```text
输出：
10 <-> 20 <-> 30 <-> 40 <-> 50 <-> NULL
```

**逐步图解（创建过程）：**

第 1 步：创建第一个节点

```text
创建第一个节点:
  head=[10], tail=[10]
  [10].prev == nullptr, [10].next == nullptr
```

第 2 步：插入值为 20 的节点

```text
插入节点 20:
  head → [10] ⇄ [20] ← tail
```

第 3 步：插入值为 30 的节点

```text
插入节点 30:
  head → [10] ⇄ [20] ⇄ [30] ← tail
```

完成后的完整结构：

```text
完成后的完整结构:
  head → [10] ⇄ [20] ⇄ [30] ⇄ [40] ⇄ [50] ← tail
```

**复杂度：** O(n)，遍历数组一次。

---

### 20. 双向链表的基本操作

#### 20.1 双向遍历

双向链表可以正向遍历（从 `head` 到 `tail`）和反向遍历（从 `tail` 到 `head`）。

**正向遍历代码：**

In [ ]:
// 正向遍历：从 head 到 tail

In [ ]:
void traverseForward(DNode* head) {
    cout << "正向遍历：";
    DNode* curr = head;
    while (curr != nullptr) {
        cout << curr->data << " -> ";
        curr = curr->next;
    }
    cout << "NULL" << endl;
}

**反向遍历代码：**

In [ ]:
// 反向遍历：从 tail 到 head

In [ ]:
void traverseBackward(DNode* tail) {
    cout << "反向遍历：";
    DNode* curr = tail;
    while (curr != nullptr) {
        cout << curr->data << " -> ";
        curr = curr->prev;
    }
    cout << "NULL" << endl;
}

**如何获取 tail？** 如果只有 `head`，需要先走到尾部：

In [ ]:
DNode* getTail(DNode* head) {
    if (head == nullptr) return nullptr;
    DNode* curr = head;
    while (curr->next != nullptr) {
        curr = curr->next;
    }
    return curr;
}

**完整示例：**

In [ ]:
int main() {
    int arr[] = {10, 20, 30, 40};
    DNode* head = createDoublyList(arr, 4);

    traverseForward(head);              // 10 -> 20 -> 30 -> 40 -> NULL
    traverseBackward(getTail(head));    // 40 -> 30 -> 20 -> 10 -> NULL

    deleteDoublyList(head);
    return 0;
}

In [ ]:
main();

```text
输出:
  正向遍历：10 -> 20 -> 30 -> 40 -> NULL
  反向遍历：40 -> 30 -> 20 -> 10 -> NULL

  正向: head → [10] → [20] → [30] → [40] → nullptr
  反向: tail → [40] → [30] → [20] → [10] → nullptr
```

**复杂度：** 正向和反向遍历都是 O(n)。

#### 20.2 在指定节点之后插入

**思路分析：**

在节点 `pos` 之后插入新节点，需要修改 4 根指针：

1. `newNode->next = pos->next`（新节点的 next 指向 pos 的下一个节点）
2. `newNode->prev = pos`（新节点的 prev 指向 pos）
3. 如果 `pos->next` 不是 `nullptr`，则 `pos->next->prev = newNode`（pos 的下一个节点的 prev 指向新节点）
4. `pos->next = newNode`（pos 的 next 指向新节点）

**完整代码：**

In [ ]:
// 在指定节点之后插入

In [ ]:
void insertAfter(DNode* pos, int val) {
    if (pos == nullptr) {
        cout << "指定节点为空" << endl;
        return;
    }

    DNode* newNode = new DNode(val);

    // 步骤 1：新节点 next 指向 pos 的下一个节点
    newNode->next = pos->next;

    // 步骤 2：新节点 prev 指向 pos
    newNode->prev = pos;

    // 步骤 3：如果 pos 不是尾节点，更新 pos 原下一个节点的 prev
    if (pos->next != nullptr) {
        pos->next->prev = newNode;
    }

    // 步骤 4：pos 的 next 指向新节点
    pos->next = newNode;
}

**详细图解：**

初始状态，在节点 20 之后插入 25：

```text
初始状态:
  head → [10] ⇄ [20] ⇄ [30] ← tail
  目标是在 pos([20]) 之后插入 newNode([25])。
```

创建新节点：

```text
创建新节点:
  newNode=[25], 其 prev 和 next 均为 nullptr。
```

步骤 1：`newNode->next = pos->next`（25 的 next 指向 30）

```text
步骤 1: newNode->next = pos->next
  newNode 记住 pos 原来的后继 [30]。
  newNode → [25] → [30]
```

步骤 2：`newNode->prev = pos`（25 的 prev 指向 20）

```text
步骤 2: newNode->prev = pos
  newNode 的 prev 指向 pos。
  [20] ← [25] → [30]
```

步骤 3：`pos->next->prev = newNode`（30 的 prev 指向 25）

```text
步骤 3: oldNext->prev = newNode
  原后继 [30] 的 prev 指向新节点 [25]。
  [25] ⇄ [30]
```

步骤 4：`pos->next = newNode`（20 的 next 指向 25）

```text
步骤 4: pos->next = newNode

  head → [10] ⇄ [20] ⇄ [25] ⇄ [30] ← tail
                ↑       ↑
               pos   newNode

含义:
  四条边都接好，新节点正式进入链表。
```

最终状态：

```text
插入后:
  head → [10] ⇄ [20] ⇄ [25] ⇄ [30] ← tail
```

**复杂度：** O(1)，给定节点指针后，插入操作是常数时间。单链表在“已给定节点后插入”时同样是 O(1)；双向链表的主要优势体现在“在给定节点之前插入”和“删除给定节点”时能直接拿到前驱。

#### 20.3 在指定节点之前插入

**思路分析：**

在节点 `pos` 之前插入新节点。如果 `pos->prev` 存在，可以利用它直接操作。

**完整代码：**

In [ ]:
// 在指定节点之前插入
// 返回可能更新的 head（如果在 head 之前插入，head 会变）

In [ ]:
DNode* insertBefore(DNode* head, DNode* pos, int val) {
    if (pos == nullptr) {
        cout << "指定节点为空" << endl;
        return head;
    }

    DNode* newNode = new DNode(val);

    // 步骤 1：新节点 next 指向 pos
    newNode->next = pos;

    // 步骤 2：新节点 prev 指向 pos 的前一个节点
    newNode->prev = pos->prev;

    // 步骤 3：如果 pos 不是头节点，更新 pos 原前一个节点的 next
    if (pos->prev != nullptr) {
        pos->prev->next = newNode;
    } else {
        // pos 是头节点，新节点成为新 head
        head = newNode;
    }

    // 步骤 4：pos 的 prev 指向新节点
    pos->prev = newNode;

    return head;
}

**复杂度：** O(1)，给定节点指针后是常数时间。

#### 20.4 删除指定节点

**双向链表删除的巨大优势：** 给定节点指针 `pos`，可通过 `pos->prev` 直接找到前驱，无需从头遍历。单链表删除给定节点需 O(n)，双向链表只需 O(1)。

**完整代码：**

In [ ]:
// 删除指定节点
// 返回可能更新的 head

In [ ]:
DNode* deleteNode(DNode* head, DNode* pos) {
    if (pos == nullptr) return head;

    // 如果 pos 不是头节点，让前驱的 next 指向后继
    if (pos->prev != nullptr) {
        pos->prev->next = pos->next;
    } else {
        // pos 是头节点，更新 head
        head = pos->next;
    }

    // 如果 pos 不是尾节点，让后继的 prev 指向前驱
    if (pos->next != nullptr) {
        pos->next->prev = pos->prev;
    }

    delete pos;
    return head;
}

**图解（删除节点 20）：**

删除前：

```text
初始状态:
  head → [10] ⇄ [20] ⇄ [30] ← tail
  目标: 删除 pos([20])，把 [10] 和 [30] 连起来。
```

步骤 1：`pos->prev->next = pos->next`（10 的 next 指向 30）

```text
步骤 1: pos->prev->next = pos->next
  next 方向跳过 [20]: [10] → [30]
  prev 尚未更新: [30] → [20]
```

步骤 2：`pos->next->prev = pos->prev`（30 的 prev 指向 10）

```text
步骤 2: pos->next->prev = pos->prev
  next: [10] → [30], prev: [30] → [10]
  pos [20] 彻底脱离链表，等待 delete。
```

释放节点 20，最终：

```text
删除完成:
  head → [10] ⇄ [30] ← tail
```

**复杂度：** O(1)，给定节点指针后是常数时间。

#### 20.5 双向链表 vs 单链表对比表

| 对比项 | 单链表 | 双向链表 | 记忆点 |
| ------ | ------ | -------- | ------ |
| 指针字段 | `next` | `prev` + `next` | 双向多一条回路 |
| 内存开销 | 较小 | 较大 | 用空间换操作便利 |
| 正向遍历 | 支持 | 支持 | 都能向后走 |
| 反向遍历 | 不支持 | 支持 | 依赖 `prev` |
| 删除给定节点 | 需找前驱 | O(1) | 双向链表自带前驱 |
| 在节点前插入 | 需找前驱 | O(1) | `prev` 直接可用 |
| 在节点后插入 | O(1) | O(1) | 都容易 |
| 代码复杂度 | 较低 | 较高 | 多维护 `prev` |
| 适合场景 | 单向扫描 | 双向移动/频繁删除 | 看是否需要前驱 |

---

### 21. 双向链表反转

双向链表的反转思路：**交换每个节点的 `prev` 和 `next` 指针，最后把 `head` 更新为原来的尾节点。**

**思路分析：**

1. 遍历每个节点
2. 对每个节点，交换 `prev` 和 `next`
3. 每轮用 `newHead` 记录最后一个处理过的节点
4. 遍历结束时 `curr == nullptr`，`newHead` 正好是原尾节点，返回它作为新 `head`

**完整代码：**

In [ ]:
// 双向链表反转

In [ ]:
DNode* reverse(DNode* head) {
    if (head == nullptr) return nullptr;

    DNode* curr = head;
    DNode* newHead = nullptr;

    while (curr != nullptr) {
        // 交换当前节点的 prev 和 next
        DNode* temp = curr->prev;
        curr->prev = curr->next;
        curr->next = temp;

        // 记录新 head（最后一个非空节点）
        newHead = curr;

        // 沿原来的 next 方向前进（交换后是 curr->prev）
        curr = curr->prev;
    }

    return newHead;
}

**逐步图解：**

初始状态：`10 <-> 20 <-> 30`

```text
初始状态:
  head → [10] ⇄ [20] ⇄ [30] ← tail
```

第 1 次循环，`curr = 10`：

```text
处理节点 [10]:
  交换前后: next 变为 nullptr, prev 变为 [20]
```

```text
步骤 1 结束:
  [10] → nullptr (其 prev 指向 20)
  curr 沿 curr->prev 前进到 [20]。
```

第 2 次循环，`curr = 20`：

```text
处理节点 [20]:
  交换前后: prev 变为 [30], next 变为 [10]
```

```text
步骤 2 结束:
  [20] ⇄ [10] → nullptr
  curr 沿 curr->prev 前进到 [30]。
```

第 3 次循环，`curr = 30`：

```text
处理节点 [30]:
  交换前后: prev 变为 nullptr, next 变为 [20]
```

```text
步骤 3 结束，反转完成:
  newHead → [30] ⇄ [20] ⇄ [10]
  curr 变为 nullptr，循环结束。
```

最终结果：

```text
反转后:
  newHead → [30] ⇄ [20] ⇄ [10] ← tail
  [30].prev == nullptr, [10].next == nullptr

含义:
  原尾节点 [30] 成为新的 head。
```

链表成功反转：`30 <-> 20 <-> 10`。

**复杂度：** O(n)，遍历每个节点一次。空间复杂度 O(1)，只用了常数额外空间。

---

### 22. 双向循环链表

#### 22.1 概念与内存结构

双向循环链表结合了双向链表和循环链表的特点：

- 每个节点有 `prev` 和 `next` 两个指针
- **尾节点的 `next` 指回头节点**（而不是 `nullptr`）
- **头节点的 `prev` 指向尾节点**（而不是 `nullptr`）

形成双向闭环。

**结构图：**

```text
结构与闭环:
  head → [10] ⇄ [20] ⇄ [30]
  [30].next == [10] (尾接头)
  [10].prev == [30] (头接尾)
```

简化示意：

```text
闭环回路:
  next: head → [10] → [20] → [30] → [10]
  prev: head → [10] ← [20] ← [30] ← [10]
```

**空双向循环链表：** `head == nullptr`。

**节点结构体：** 与双向链表相同，使用 `DNode`。

#### 22.2 基本操作（插入、删除、遍历）

**创建双向循环链表（尾插法）：**

In [ ]:
// 创建双向循环链表

In [ ]:
DNode* createDoublyCircularList(int arr[], int n) {
    if (n <= 0) return nullptr;

    DNode* head = new DNode(arr[0]);
    DNode* tail = head;

    for (int i = 1; i < n; i++) {
        DNode* newNode = new DNode(arr[i]);
        tail->next = newNode;
        newNode->prev = tail;
        tail = newNode;
    }

    // 形成双向循环
    tail->next = head;  // 尾节点 next 指回头节点
    head->prev = tail;  // 头节点 prev 指向尾节点

    return head;
}

**图解（创建 10 <-> 20 <-> 30 双向循环链表）：**

最后一步 `tail->next = head; head->prev = tail;`：

```text
首尾相连形成闭环:
  tail->next = head; (30.next = 10)
  head->prev = tail; (10.prev = 30)
  结果: [10] ⇄ [20] ⇄ [30] 闭环。
```

**正向遍历（迭代）：**

In [ ]:
// 双向循环链表正向遍历

In [ ]:
void displayForward(DNode* head) {
    if (head == nullptr) {
        cout << "链表为空" << endl;
        return;
    }

    DNode* curr = head;
    do {
        cout << curr->data << " <-> ";
        curr = curr->next;
    } while (curr != head);  // 回到 head 时停止

    cout << "(回到 " << head->data << ")" << endl;
}

**反向遍历（迭代）：**

In [ ]:
// 双向循环链表反向遍历

In [ ]:
void displayBackward(DNode* head) {
    if (head == nullptr) {
        cout << "链表为空" << endl;
        return;
    }

    // 反向遍历从 tail 开始，即 head->prev
    DNode* tail = head->prev;
    DNode* curr = tail;

    do {
        cout << curr->data << " <-> ";
        curr = curr->prev;
    } while (curr != tail);  // 回到 tail 时停止

    cout << "(回到 " << tail->data << ")" << endl;
}

**在头部插入节点：**

In [ ]:
// 在双向循环链表头部插入

In [ ]:
DNode* insertAtHead_DC(DNode* head, int val) {
    DNode* newNode = new DNode(val);

    // 空链表
    if (head == nullptr) {
        newNode->next = newNode;
        newNode->prev = newNode;
        return newNode;
    }

    DNode* tail = head->prev;  // 通过 head->prev 直接得到尾节点，O(1)！

    // 新节点链接到 head 和 tail 之间
    newNode->next = head;
    newNode->prev = tail;
    head->prev = newNode;
    tail->next = newNode;

    return newNode;  // 新节点成为新 head
}

**图解（在头部插入 5）：**

插入前（双向循环链表：10 <-> 20 <-> 30）：

```text
初始状态:
  head → [10] ⇄ [20] ⇄ [30] ← tail
  通过 head->prev 直接获得 tail [30]。
```

插入后（5 <-> 10 <-> 20 <-> 30）：

```text
完成插入:
  新节点 [5] 插在 tail 和旧 head 之间，并更新为新 head。
  head → [5] ⇄ [10] ⇄ [20] ⇄ [30] ← tail
```

注意：双向循环链表在头部插入只需 O(1)，因为通过 `head->prev` 直接就能找到尾节点，不需要遍历。

**删除指定节点：**

In [ ]:
// 删除双向循环链表中的指定节点
// 前提：pos 必须属于以 head 为入口的这条链表。
// 这个函数更适合作为封装类的 private helper；公共接口应先查找/验证 pos。

In [ ]:
DNode* deleteNode_DC(DNode* head, DNode* pos) {
    if (head == nullptr) return nullptr;
    if (pos == nullptr) return head;

    // 只有一个节点
    if (pos->next == pos) {
        delete pos;
        return nullptr;
    }

    // 更新前驱和后继的指针
    pos->prev->next = pos->next;
    pos->next->prev = pos->prev;

    // 如果删除的是 head，需要更新 head
    DNode* newHead = (pos == head) ? pos->next : head;

    delete pos;
    return newHead;
}

**图解（删除节点 20）：**

删除前：

```text
初始状态:
  head → [10] ⇄ [20] ⇄ [30] ← tail
  目标: 删除 pos([20])，让 [10] 和 [30] 连起来。
```

执行 `pos->prev->next = pos->next`（10 的 next 指向 30）和 `pos->next->prev = pos->prev`（30 的 prev 指向 10）：

```text
修改指针，摘除节点:
  [10].next = [30]
  [30].prev = [10]
  [20] 被摘出，等待 delete。
```

释放节点 20，最终：

```text
删除完成:
  head → [10] ⇄ [30] ← tail
```

即：`[10] <-> [30]` 双向循环。

**复杂度汇总：**

| 操作 | 时间复杂度 | 依赖条件 |
| ---- | ---------- | -------- |
| 创建 | O(n) | 遍历输入数组 |
| 正向遍历 | O(n) | 沿 `next` 一圈 |
| 反向遍历 | O(n) | 沿 `prev` 一圈 |
| 头部插入 | O(1) | `head->prev` 找尾 |
| 尾部插入 | O(1) | `head->prev` 找尾 |
| 删除指定节点 | O(1) | 已给定节点指针 |
| 查找某值 | O(n) | 最坏走一圈 |

双向循环链表在头部/尾部操作上的 O(1) 特性，使其成为实现 **deque（双端队列）** 等数据结构的理想底层结构。


---

## 第七阶段：模板化、迭代器与 STL std::list

> 前面我们用函数操作裸指针、用类封装了链表。本阶段将其**模板化**（支持任意类型）、加入**迭代器**（支持 range for），然后过渡到 STL `std::list`。

> 注意：第 14 章的 `LinkedList` 是“单链表封装版”，重点是 RAII 和隐藏 `head`。第 23 章开始改用“双向循环链表 + 哨兵节点”模型，更接近 `std::list`，便于统一处理 `begin()/end()`、头尾插入和空表边界。

---

### 23. 模板化链表

#### 23.1 为什么需要模板化

前面的 `LinkedList` 只能存 `int`。若想存 `double`、`string` 或自定义类型，C++ 的**模板（Template）**可以一次编写、支持任意类型。

#### 23.2 使用 `template<typename T>` 改造链表

In [ ]:
#include <initializer_list>
#include <cstddef>
#include <iostream>
#include <stdexcept>
#include <string>
#include <utility>

In [ ]:
template<typename T>
class LinkedList {
private:
    struct Node {
        T data;             // 数据域变为泛型 T
        Node* prev;
        Node* next;
        Node(const T& val = T()) : data(val), prev(this), next(this) {}
    };

    Node* sentinel;
    std::size_t size_;

    // 核心辅助函数保持不变，只是 T 代替了 int
    void insertBefore(Node* pos, const T& val) {
        Node* newNode = new Node(val);
        Node* prevNode = pos->prev;
        prevNode->next = newNode;
        newNode->prev = prevNode;
        newNode->next = pos;
        pos->prev = newNode;
        ++size_;
    }

    void eraseNode(Node* pos) {
        pos->prev->next = pos->next;
        pos->next->prev = pos->prev;
        delete pos;
        --size_;
    }

    void clearNodes() {
        Node* curr = sentinel->next;
        while (curr != sentinel) {
            Node* temp = curr->next;
            delete curr;
            curr = temp;
        }
        sentinel->next = sentinel;
        sentinel->prev = sentinel;
        size_ = 0;
    }

public:
    LinkedList() : size_(0) { sentinel = new Node(); }

    LinkedList(std::initializer_list<T> init) : LinkedList() {
        for (const T& val : init) pushBack(val);
    }

    LinkedList(const LinkedList& other) : LinkedList() {
        Node* curr = other.sentinel->next;
        while (curr != other.sentinel) {
            pushBack(curr->data);
            curr = curr->next;
        }
    }

    LinkedList& operator=(LinkedList other) {
        swap(other);
        return *this;
    }

    void swap(LinkedList& other) {
        std::swap(sentinel, other.sentinel);
        std::swap(size_, other.size_);
    }

    ~LinkedList() { clearNodes(); delete sentinel; }

    void pushFront(const T& val) { insertBefore(sentinel->next, val); }
    void pushBack(const T& val)  { insertBefore(sentinel, val); }
    void popFront() { if (!empty()) eraseNode(sentinel->next); }
    void popBack()  { if (!empty()) eraseNode(sentinel->prev); }

    T& front() {
        if (empty()) throw std::out_of_range("front() on empty list");
        return sentinel->next->data;
    }
    T& back() {
        if (empty()) throw std::out_of_range("back() on empty list");
        return sentinel->prev->data;
    }
    const T& front() const {
        if (empty()) throw std::out_of_range("front() on empty list");
        return sentinel->next->data;
    }
    const T& back() const {
        if (empty()) throw std::out_of_range("back() on empty list");
        return sentinel->prev->data;
    }

    bool empty() const { return sentinel->next == sentinel; }
    std::size_t size() const { return size_; }
    void clear() { clearNodes(); }

    void print() const {
        Node* curr = sentinel->next;
        std::cout << "[";
        while (curr != sentinel) {
            std::cout << curr->data;
            if (curr->next != sentinel) std::cout << ", ";
            curr = curr->next;
        }
        std::cout << "]" << std::endl;
    }
};

**哨兵节点模型：**

```text
哨兵节点模型:
  空链表: sentinel ⇄ sentinel (自闭环)
  非空链表: begin() 指向首元素，end() 指向 sentinel。
  左右 sentinel 是同一个物理节点。
```

**插入只改局部四根指针：**

| 步骤 | 指针更新 | 结果 |
| ---- | -------- | ---- |
| 1 | `prevNode->next = newNode` | 前驱接新节点 |
| 2 | `newNode->prev = prevNode` | 新节点接前驱 |
| 3 | `newNode->next = pos` | 新节点接 `pos` |
| 4 | `pos->prev = newNode` | `pos` 接新节点 |

删除 `pos` 时只需要两步：

| 步骤 | 指针更新 | 结果 |
| ---- | -------- | ---- |
| 1 | `pos->prev->next = pos->next` | 前驱跳过 `pos` |
| 2 | `pos->next->prev = pos->prev` | 后继跳过 `pos` |

#### 23.3 使用示例

下面示例依赖上一节的 `LinkedList<T>` 类定义。若要复制成单个可编译文件，需要把 23.2 的类定义放在这个 `main` 函数之前。

In [ ]:
int main() {
    // 存 int
    LinkedList<int> intList = {1, 2, 3, 4, 5};
    intList.print();  // [1, 2, 3, 4, 5]

    // 存 double
    LinkedList<double> doubleList = {1.1, 2.2, 3.3};
    doubleList.print();  // [1.1, 2.2, 3.3]

    // 存 string
    LinkedList<std::string> strList = {"Hello", "World", "C++"};
    strList.pushBack("Template");
    strList.print();  // [Hello, World, C++, Template]

    // 拷贝
    LinkedList<int> copy(intList);
    copy.pushBack(6);
    copy.print();  // [1, 2, 3, 4, 5, 6]
    intList.print();  // [1, 2, 3, 4, 5]  深拷贝互不影响

    return 0;
}

In [ ]:
main();

> **限制说明**：这个简化模板把哨兵节点建模为 `Node<T>`，使用 `Node(const T& val = T())` 初始化哨兵，因此要求 `T` 可默认构造。工业级 `std::list` 会把哨兵节点和数据节点分离来避免这个限制。

> **异常安全说明**：这个版本仍直接使用 `new/delete`，用于教学理解。若复制构造过程中 `pushBack` 抛异常，需要更完整的 RAII/allocator 设计来清理已经构造的节点；工业级容器会处理这些边界。

#### 23.4 模板类的文件组织

**关键规则：模板类的声明和实现通常都放在头文件中（`.h` / `.hpp`）。**

原因是模板在**编译期实例化**。编译器看到 `LinkedList<int>` 时，需要能看到完整的实现代码。如果实现放在 `.cpp` 里，其他翻译单元看不到实现，会报链接错误。

In [ ]:
// 正确做法：全部放在头文件中
// LinkedList.hpp

In [ ]:
template<typename T>
class LinkedList {
    // 声明 + 实现全部在这里
};

In [ ]:
// 错误做法：声明和实现分离
// LinkedList.h   -> 只有声明
// LinkedList.cpp -> 实现（其他 .cpp 文件 include 后找不到实现）

---

### 24. 自定义迭代器

#### 24.1 什么是迭代器？为什么需要它？

**迭代器（Iterator）**是一种设计模式，提供**统一的方式**遍历容器，无需暴露内部实现。

**没有迭代器时：**

In [ ]:
// 遍历数组 -- 用下标

In [ ]:
for (int i = 0;
i < arr.size();
++i)
    std::cout << arr[i];
// 遍历链表 -- 用指针
for (Node* p = head;
p != nullptr;
p = p->next)
    std::cout << p->data;
// 每种容器的遍历方式都不一样！

**有了迭代器：**

In [ ]:
// 不管是数组、链表、树还是哈希表，统一写法：

In [ ]:
for (auto it = container.begin();
it != container.end();
++it)
    std::cout << *it;
// 甚至可以用范围 for（编译器自动翻译为迭代器版本）
for (auto& item : container)
    std::cout << item;

**迭代器的本质**：像一个"智能指针"，通过重载 `*`、`->`、`++`、`--`、`==`、`!=` 运算符，让遍历用起来和指针一样自然。

#### 24.2 为链表实现迭代器

下面代码展示的是 **23.2 的 `LinkedList<T>` 加入迭代器后的类骨架**，只保留和迭代器直接相关的成员。不要把这段和 23.2 的完整类定义并列复制，否则会重复定义 `LinkedList`；实际使用时，应把这里的 `iterator`、`const_iterator`、`begin/end`、`insert/erase` 合并进 23.2 的同一个类中。

In [ ]:
template<typename T>
class LinkedList {
private:
    struct Node {
        T data;
        Node *prev, *next;
        Node(const T& val = T()) : data(val), prev(this), next(this) {}
    };
    Node* sentinel;
    std::size_t size_;

public:
    class iterator {
    private:
        Node* ptr;
        friend class LinkedList;
    public:
        iterator(Node* p = nullptr) : ptr(p) {}
        T& operator*() { return ptr->data; }
        T* operator->() { return &(ptr->data); }
        iterator& operator++() { ptr = ptr->next; return *this; }
        iterator operator++(int) { iterator tmp = *this; ptr = ptr->next; return tmp; }
        iterator& operator--() { ptr = ptr->prev; return *this; }
        iterator operator--(int) { iterator tmp = *this; ptr = ptr->prev; return tmp; }
        bool operator==(const iterator& other) const { return ptr == other.ptr; }
        bool operator!=(const iterator& other) const { return ptr != other.ptr; }
    };

    class const_iterator {
    private:
        const Node* ptr;
        friend class LinkedList;
    public:
        const_iterator(const Node* p = nullptr) : ptr(p) {}
        const_iterator(const iterator& it) : ptr(it.ptr) {}
        const T& operator*() const { return ptr->data; }
        const T* operator->() const { return &(ptr->data); }
        const_iterator& operator++() { ptr = ptr->next; return *this; }
        const_iterator operator++(int) { const_iterator tmp = *this; ptr = ptr->next; return tmp; }
        const_iterator& operator--() { ptr = ptr->prev; return *this; }
        const_iterator operator--(int) { const_iterator tmp = *this; ptr = ptr->prev; return tmp; }
        bool operator==(const const_iterator& other) const { return ptr == other.ptr; }
        bool operator!=(const const_iterator& other) const { return ptr != other.ptr; }
    };

    iterator begin() { return iterator(sentinel->next); }
    iterator end()   { return iterator(sentinel); }
    const_iterator begin() const { return const_iterator(sentinel->next); }
    const_iterator end()   const { return const_iterator(sentinel); }
    const_iterator cbegin() const { return const_iterator(sentinel->next); }
    const_iterator cend()   const { return const_iterator(sentinel); }

    iterator insert(iterator pos, const T& val) {
        // pos 必须是本链表中的有效迭代器，可以等于 end()
        Node* newNode = new Node(val);
        Node* posNode = pos.ptr;
        newNode->prev = posNode->prev; newNode->next = posNode;
        posNode->prev->next = newNode; posNode->prev = newNode;
        ++size_;
        return iterator(newNode);
    }

    iterator erase(iterator pos) {
        // pos 必须是本链表中的有效迭代器；erase(end()) 没有元素可删
        Node* posNode = pos.ptr;
        if (posNode == sentinel) return end();
        Node* nextNode = posNode->next;
        posNode->prev->next = nextNode;
        nextNode->prev = posNode->prev;
        delete posNode;
        --size_;
        return iterator(nextNode);
    }
};

若希望复制成单个可编译文件，做法是：以 23.2 的完整类为主体，把本节新增成员放入同一个 `public` 区域，并保留 23.2 已有的构造、析构、`pushBack`、`clear` 等成员。

#### 24.3 `begin()` 和 `end()` 的设计哲学

```text
begin() 和 end() 范围表示:
  非空链表: begin() -> [10], end() -> sentinel
  有效范围: [begin, end) 包含 [10], [20], [30]
  空链表: begin() == end() == sentinel
```

- `begin()` 返回指向**第一个数据节点**的迭代器
- `end()` 返回指向**哨兵节点**的迭代器（"越过最后一个元素的位置"）
- 遍历条件：`it != end()`，到达哨兵节点时说明遍历完毕
- 空链表时 `sentinel->next == sentinel`，所以 `begin() == end()`

这种**左闭右开**的设计是 STL 的核心约定，所有容器都遵循。

#### 24.4 使用示例

这个示例默认你已经有 **23.2 的完整类实现**，并且加上了 **24.2 的迭代器代码**。

In [ ]:
int main() {
    LinkedList<int> list = {10, 20, 30, 40, 50};

    // 迭代器遍历
    for (auto it = list.begin(); it != list.end(); ++it) {
        std::cout << *it << " ";
    }
    std::cout << std::endl;  // 10 20 30 40 50

    // 范围 for（编译器自动翻译为迭代器版本）
    for (int& val : list) {
        val *= 2;
    }
    for (int val : list) {
        std::cout << val << " ";
    }
    std::cout << std::endl;  // 20 40 60 80 100

    // 用迭代器插入
    auto it = list.begin();
    ++it;              // 指向第 2 个元素（40）
    list.insert(it, 35);  // 在 40 前面插入 35

    // 用迭代器删除
    it = list.begin();
    ++it;              // 指向 35
    it = list.erase(it);  // 删除 35，it 现在指向 40
    // erase 返回下一个有效迭代器，一定要接住！

    // 存储结构体
    struct Student {
        std::string name;
        int score;
    };

    LinkedList<Student> students;
    students.pushBack({"Alice", 95});
    students.pushBack({"Bob", 87});

    for (auto it = students.begin(); it != students.end(); ++it) {
        std::cout << it->name << ": " << it->score << std::endl;
        // 箭头运算符 -> 直接访问 Student 的成员
    }

    return 0;
}

In [ ]:
main();

> **注意**：`erase` 之后原迭代器**失效**（节点已被删除）。必须用 `it = list.erase(it)` 接住返回值。

> **简化说明**：这里的迭代器足够支持遍历、范围 for、插入和删除，但还没有补全 `iterator_category`、`difference_type` 等 STL 迭代器类型别名，因此不算完整的标准库风格迭代器。

---

### 25. 过渡到 STL `std::list`

现在你已亲手实现了链表和迭代器，过渡到 `std::list` 会非常顺畅——你写的就是 `std::list` 的简化版。

In [ ]:
#include <list>
#include <forward_list>
#include <iostream>
#include <algorithm>
#include <functional>
#include <utility>

#### 25.0 单链表对应 `std::forward_list`

标准库里有两个链表容器：

| 容器 | 底层模型 | 主要特点 |
| ---- | -------- | -------- |
| `std::forward_list` | 单链表 | 只支持向前遍历，内存开销较低 |
| `std::list` | 双向链表 | 支持双向遍历，给定位置插入/删除更方便 |

`std::forward_list` 更接近本教程前半部分的单链表。它没有 `size()`（为了保持轻量）、没有反向迭代器，插入删除通常围绕“前驱位置”进行：

In [ ]:
std::forward_list<int> fl = {10, 20, 30};
fl.push_front(5);
// 头插

auto beforeFirst = fl.before_begin();
fl.insert_after(beforeFirst, 1);
// 在第一个元素之前插入，本质是 after 前驱

auto it = fl.begin();
// 指向 1
fl.erase_after(it);
// 删除 it 后面的元素

如果你只需要单向遍历并且非常在意每个节点的指针开销，可以考虑 `std::forward_list`；如果需要前后移动、尾部操作或更接近双向循环链表模型，使用 `std::list`。

#### 25.1 构造函数

In [ ]:
// 1. 默认构造：空链表

In [ ]:
std::list<int> l1;
// 2. 指定个数和初始值
std::list<int> l2(5, 100);
// {100, 100, 100, 100, 100}

// 3. 初始化列表
std::list<int> l3 = {1, 2, 3, 4, 5};
// 4. 拷贝构造
std::list<int> l4(l3);
// 5. 迭代器范围构造
std::list<int> l5(l3.begin(), l3.end());
// 6. 移动构造（C++11）
std::list<int> l6(std::move(l5));
// l6 获得数据，l5 仍有效但内容状态不应再假定

#### 25.2 赋值与大小

In [ ]:
std::list<int> l1 = {1, 2, 3};
std::list<int> l2;
l2 = l1;
l2.assign(5, 10);
// {10, 10, 10, 10, 10}
l2.assign({7, 8, 9});
// {7, 8, 9}

l1.size();
// 3
l1.empty();
// false
l1.resize(5);
// {1, 2, 3, 0, 0}  不足补 0
l1.resize(7, 99);
// {1, 2, 3, 0, 0, 99, 99}  不足补 99
l1.resize(2);
// {1, 2}  多余截断
// 注意：std::list 没有 capacity()，因为链表不需要预分配

#### 25.3 插入与删除

In [ ]:
std::list<int> l = {10, 20, 30};
// 头尾操作
l.push_back(40);
// {10, 20, 30, 40}
l.push_front(5);
// {5, 10, 20, 30, 40}
l.pop_back();
// {5, 10, 20, 30}
l.pop_front();
// {10, 20, 30}

l.emplace_back(40);
// 原地构造，某些场景可避免临时对象
l.emplace_front(5);
// 迭代器插入
auto it = l.begin();
std::advance(it, 2);
// 移动迭代器到第 3 个位置
l.insert(it, 25);
// 在 it 之前插入 25
l.insert(it, 3, 99);
// 在 it 之前插入 3 个 99
l.insert(it, {61, 62});
// 在 it 之前插入列表

// 删除
it = l.begin();
it = l.erase(it);
// 删除 it 指向的元素，返回下一个迭代器
l.erase(l.begin(), l.end());
// 范围删除

l.remove(10);
// 删除所有值为 10 的元素
l.remove_if([](int x) { return x > 50; });
// 条件删除

l.clear();
// 清空

> **迭代器失效规则**：`erase` 会使被删除元素对应的迭代器失效，所以要用返回值接住下一个位置；其他未删除元素的迭代器通常保持有效。

#### 25.4 数据访问与遍历

In [ ]:
std::list<int> l = {10, 20, 30};
l.front();
// 10
l.back();
// 30

// 注意：list 不支持 [] 和 at()，因为不支持随机访问！
// l[0];    // 编译错误！
// l.at(0); // 编译错误！

// 迭代器遍历
for (auto it = l.begin();
it != l.end();
++it)
    std::cout << *it << " ";
// 范围 for
for (int val : l)
    std::cout << val << " ";
// 反向遍历
for (auto rit = l.rbegin();
rit != l.rend();
++rit)
    std::cout << *rit << " ";
// 输出：30 20 10

#### 25.5 `std::list` 特有的算法操作

`std::list` 提供成员函数版本的算法，比通用 `<algorithm>` 更高效（只修改指针，不搬移元素）。

In [ ]:
// ===== sort() =====
// list 不能用 std::sort()（不支持随机访问迭代器），必须用成员函数

In [ ]:
std::list<int> l = {5, 3, 1, 4, 2};
l.sort();
// 升序：{1, 2, 3, 4, 5}
l.sort(std::greater<int>());
// 降序：{5, 4, 3, 2, 1}

// ===== reverse() =====
l.reverse();
// ===== unique() =====
// 删除连续重复元素（通常先 sort 再 unique）
std::list<int> l2 = {1, 1, 2, 2, 3, 3, 3};
l2.unique();
// {1, 2, 3}

// ===== merge() =====
// 合并另一个有序链表（两个都必须已按同一规则排序）
std::list<int> a = {1, 3, 5};
std::list<int> b = {2, 4, 6};
a.merge(b);
// a = {1, 2, 3, 4, 5, 6}，b 变空
// 底层只修改指针，O(n)，不复制元素

// ===== splice() =====
// 将另一个链表的元素"移接"到当前链表（O(1)！只改指针）
std::list<int> x = {1, 2, 3};
std::list<int> y = {10, 20, 30};
auto pos = x.begin();
++pos;
// 指向 2

x.splice(pos, y);
// y 的所有元素移到 x 中 pos 之前
// x = {1, 10, 20, 30, 2, 3}
// y = {}（元素被移走了，不是复制！）

// splice 也可以只移动单个元素或一段范围
std::list<int> m = {100, 200, 300};
x.splice(x.begin(), m, m.begin());
// 只把 m 的第一个元素 100 移接到 x 头部

> **`splice` 是 `list` 最强大的操作之一**：整表和单节点移接通常为 O(1)，区间移接为 O(k)。
> 被移动的节点不会被复制或销毁，原迭代器仍有效，只是所属容器变了。
> 使用 `merge` / `splice` 时，两个 `std::list` 的分配器必须兼容；初学阶段使用默认分配器即可满足这一点。

---

### 26. STL `std::list` 源码简析（选学）

> 帮助理解 `std::list` 的底层实现，和你手写的版本做对照。

#### 26.1 节点结构

GCC 的 libstdc++ 实现大体上会把 `list` 的节点分两层（不同版本细节可能不同）：

In [ ]:
// 基类：只存储前后指针（不含数据）

In [ ]:
struct _List_node_base {
    _List_node_base* _M_next;
    _List_node_base* _M_prev;
};

In [ ]:
// 派生类：添加数据域
template<typename _Tp>
struct _List_node : public _List_node_base {
    _Tp _M_data;
};

**为什么分两层？** 哨兵节点不存储数据，只需 `prev` 和 `next`，类型为 `_List_node_base`；数据节点类型为 `_List_node<T>`，多一个数据域。这样哨兵不浪费一个 `T` 的空间。

```text
两层节点设计对比:
  手写版: sentinel 是 Node<T>，data 占位浪费。
  std::list: sentinel 是 _List_node_base，无 data。
             数据节点 _List_node<T> 继承自基类，附加 _M_data。
```

#### 26.2 迭代器实现

In [ ]:
template<typename _Tp>
struct _List_iterator {
    _List_node_base* _M_node;  // 内部存储基类指针

    _Tp& operator*() const {
        // 向下转型为 _List_node<_Tp>*，访问 _M_data
        return static_cast<_List_node<_Tp>*>(_M_node)->_M_data;
    }

    _List_iterator& operator++() {
        _M_node = _M_node->_M_next;
        return *this;
    }
    // ... 其他运算符类似 ...
};

> 和你手写的迭代器几乎一模一样！区别只是用了基类指针 + `static_cast`。

#### 26.3 内存分配器（Allocator）

In [ ]:
template<typename _Tp, typename _Alloc = std::allocator<_Tp>>
class list { /* ... */ };

第二个模板参数是**分配器**，默认 `std::allocator<T>`：
- 负责内存的申请和释放，替代直接 `new` / `delete`
- 底层调用的还是 `operator new` / `operator delete`
- 可替换为自定义分配器（如内存池分配器），优化大量小对象分配的性能（链表节点恰好就是大量小对象）
- 初学者使用默认即可

---

### 总结：从手写链表到 `std::list` 的对照

**结构对应：**

| 手写概念 | `std::list` 概念 | 直观理解 |
| -------- | ----------------- | -------- |
| `Node<T>` | `_List_node<T>` | 真正存数据的节点 |
| `sentinel` | `_List_node_base` | `end()` 所在的哨兵 |
| `sentinel->next` | `_M_next` | 第一个数据节点 |
| `sentinel->prev` | `_M_prev` | 最后一个数据节点 |
| 迭代器里的指针 | `_M_node` | 当前迭代器位置 |

STL 把“链接字段”和“数据字段”拆成 `_List_node_base` 与 `_List_node<T>` 两层。这样哨兵节点只保存 `prev` / `next`，不需要额外占一个 `T`。

**操作对应：**

| 手写操作 | `std::list` 操作 | 直观理解 |
| -------- | ---------------- | -------- |
| `begin()` | `begin()` | 指向第一个元素 |
| `end()` | `end()` | 指向哨兵，不是元素 |
| `insertBefore(pos)` | `insert(pos)` | 在 `pos` 前插入 |
| `eraseNode(pos)` | `erase(pos)` | 摘除当前位置 |
| `pushBack(x)` | `push_back(x)` | 插到哨兵前面 |
| `new/delete` | `Allocator` | 分配和释放节点 |

> **关键认知**：常见 `std::list` 实现通常采用**带哨兵节点的双向链表结构 + 迭代器类**。C++ 标准规定的是接口、复杂度和迭代器性质，不强制具体内部布局；STL 实现在此基础上加入了分配器、类型特征、异常安全等工业级特性。

---

### 阶段练习建议

| 阶段 | 练习 |
| ---- | ---- |
| 基础 | 手动画出 `head`、`curr`、`tail` 在遍历和尾插时的变化 |
| 单链表核心 | 实现 `insertAtEnd`、`deleteLast`、`findMiddle`，并覆盖空表/单节点 |
| 高级操作 | 合并两个有序链表，要求不创建新节点，只重连指针 |
| 循环链表 | 实现删除指定值，并说明为什么不能用 `curr != nullptr` 停止 |
| 双向链表 | 给定节点指针删除节点，并分别测试头节点、尾节点、中间节点 |
| STL | 用 `std::forward_list` 和 `std::list` 各重写一个手写例子，对比迭代器失效规则 |

---

### 附录：链表常见面试题速查

| 题目              | 核心技巧                     | 时间复杂度 | 额外空间 |
| ----------------- | ---------------------------- | ---------- | -------- |
| 反转链表          | 三指针迭代 / 递归            | O(n)       | 迭代 O(1)，递归 O(n) |
| 检测环            | 快慢指针                     | O(n)       | O(1) |
| 找环入口          | 快慢指针 + 数学推导          | O(n)       | O(1) |
| 合并两个有序链表  | 哨兵节点 + 双指针            | O(n+m)     | O(1) |
| 找倒数第 k 个节点 | 快慢指针（快先走 k 步）      | O(n)       | O(1) |
| 链表排序          | 归并排序                     | O(n log n) | 递归版通常 O(log n) |
| 判断回文链表      | 找中点 + 反转后半段 + 比较   | O(n)       | O(1) |
| 相交链表找交点    | 双指针各走一遍对方路径       | O(n+m)     | O(1) |
| 删除重复节点      | 有序则顺序去重；无序用哈希表 | O(n)       | 有序 O(1)，无序 O(n) |